# Libraries,environment and functions

In [ ]:
!pip install kmodes

In [ ]:
!pip install chrono24

  Obtaining dependency information for chrono24 from https://files.pythonhosted.org/packages/07/b6/85d7cebc7ebd9316d480da6eb28293fc34ca373fe76dce4754dd544435b6/chrono24-0.2.10-py3-none-any.whl.metadata


In [ ]:
!pip install gower

  Obtaining dependency information for gower from https://files.pythonhosted.org/packages/99/23/88b526457ea992e0a47147a886db3d749d07347c8d3a303f6076deee7299/gower-0.1.2-py3-none-any.whl.metadata


In [ ]:
#from sklearn.ensemble import IsolationForest
#from sklearn.neighbors import LocalOutlierFactor
#import chrono24

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import gower
import sys
import cv2
import kmodes
import pandas as pd
import requests
import shutil
from bs4 import BeautifulSoup
import re
import json
import time
import xml.etree.ElementTree as ET
import logging
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import MultiLabelBinarizer
from kmodes.kprototypes import KPrototypes
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
SCRNUM=0
MULTI=False
BRND='Longines'
VARIANT='Case material'
MAXURLS=500
%matplotlib inline
logging.basicConfig(filename='chrono24_listing_download.log', level=logging.DEBUG, filemode='a')

In [ ]:
def firstcolnameendwith0(dtbs):
   for item in list(dtbs.columns):
    if item[-1]=='0':
        frst=item
        break
   return frst

In [ ]:
table_of_brandsinloops=pd.read_excel('/kaggle/input/table-of-brandsinloops/table_of_brandsinloops.xlsx',index_col=0)
if MULTI:
    lst_brn=list(table_of_brandsinloops[f'brands{SCRNUM}'])
    l=0
    if lst_brn[-1]!='Completed':
      for d in lst_brn:
       if d!='Completed':
         BRND=d
         table_of_brandsinloops.loc[l,f'brands{SCRNUM}']='Completed'
         table_of_brandsinloops.to_excel("table_of_brandsinloops.xlsx")
         break
       l=l+1
    else:
      lst_upd=list(pd.read_excel('/kaggle/input/table-of-brandsinloops/table_of_brands.xlsx',index_col=0)['brands'])
      lst_upd.remove('Rolex')
      table_of_brandsinloops.loc[:,f'brands{SCRNUM}']=lst_upd
      table_of_brandsinloops.to_excel("table_of_brandsinloops.xlsx")
      print('all brands were completed,run script once more')
      sys.exit('exit')
else:
    _=0

In [ ]:
#BRND=re.sub('& ','',BRND)

# Extraction data from raw information

In [ ]:
#a=0
#listings=[]
#for root, dirs, files in os.walk('/kaggle/input'):
# for name in files:
#  if re.search('info_',name):
#  path=os.path.join(root, name)
#   listings_exm
#   a=a+1
#    if a%100==0:
#      print(f'file: {a}')
#    file_data = json.load(f)
#    listings.append(file_data)

# Upload data

In [ ]:
with open('/kaggle/input/testmodels0-1/model_links_listings.json',encoding="utf-8") as f:
    models= json.load(f)

In [ ]:
with open('/kaggle/input/listings1/listings',encoding="utf-8") as f:
    listings= json.load(f)

In [ ]:
#with open(f'/kaggle/input/listings1/{BRND.lower()}_listings',encoding="utf-8") as f:
#    omega_listings= json.load(f)

In [ ]:
models[0:5]

[{'brand': 'Patek Philippe',
  'name': 'Sky Moon Tourbillon',
  'refnum': '6002r001',
  'url': 'https://www.chrono24.com/patekphilippe/ref-6002r001.htm?caseMaterials=1&dialColor=723',
  'modelimageurl': 'https://drive.google.com/open?id=1TCb_TpAsqXxR7Xt9NAZfsJbUcfWWPzNR',
  'listings': 6,
  'listing_urls': ['/patekphilippe/patek-philippe-sky-moon-tourbillon-6002r--neu--2022--full-set--id26075293.htm',
   '/patekphilippe/patek-philippe-sky-moon-tourbillon-6002r-001--id30715073.htm',
   '/patekphilippe/patek-philippe-sky-moon-tourbillon-6002r-001--id30703809.htm',
   '/patekphilippe/patek-philippe-sky-moon-tourbillon--id31588913.htm',
   '/patekphilippe/patek-philippe-sky-moon-tourbillon--id26936109.htm',
   '/patekphilippe/patek-philippe-grand-complications-6002r-sky-moon-tourbillon-rose-gold--id25480438.htm']},
 {'brand': 'Rolex',
  'name': '1908',
  'refnum': '52509',
  'url': 'https://www.chrono24.com/rolex/ref-52509.htm?caseMaterials=8',
  'modelimageurl': 'https://drive.google.com/

In [ ]:
#omega_listings[0:5]

# Models processing and rebuilding

In [ ]:
dbmodels=pd.json_normalize(models)
dbmodels=dbmodels.drop(['listing_urls'],axis=1)

In [ ]:
maxvll=0
maxinlist=list(pd.json_normalize(models)['listing_urls'])
for rr in maxinlist:
    if len(rr)>maxvll:
        maxvll=len(rr)

In [ ]:
dbmodels=pd.concat([dbmodels,pd.DataFrame(pd.json_normalize(models)['listing_urls'].to_list())],axis=1)

In [ ]:
dbmodels=dbmodels.set_index('url')
dbmod=dbmodels.copy()

In [ ]:
dbmodels=pd.DataFrame(np.array(dbmodels)[:,-maxvll::],columns=dbmodels.columns[-maxvll::],index=dbmodels.index)

In [ ]:
dbmodels

,0,1,2,3,4,5,6,7,8,9,...,53,54,55,56,57,58,59,60,61,62
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/patekphilippe/ref-6002r001.htm?caseMaterials=1&dialColor=723,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-grand-complicati...,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-52509.htm?caseMaterials=8,/rolex/rolex-new-release-1908-white-gold-black...,/rolex/1908--id31783339.htm,/rolex/rolex-1908-perpetual-18k-39mm-white-gol...,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-52508.htm?caseMaterials=3,/rolex/rolex-new-release-1908-watch-52508--id3...,/rolex/rolex-1908-gelbgold--2023--neu--full-se...,/rolex/rolex-new-release-1908-watch-52508-blac...,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/patekphilippe/ref-57111a018.htm?caseMaterials=4&dialColor=710,/patekphilippe/patek-philippe-nautilus--id3021...,/patekphilippe/patek-philippe-nautilus-tiffany...,/patekphilippe/patek-philippe-nautilus--id2692...,/patekphilippe/nautilus-tiffany-and-co-blue-di...,/patekphilippe/nautilus-tiffany-amp-co-dial--s...,/patekphilippe/nautilus-tiffany-amp-codial--id...,/patekphilippe/patek-philippe-nautilus--id2687...,/patekphilippe/patek-philippe-nautilus--id2702...,/patekphilippe/patek-philippe-nautilus-57111a-...,/patekphilippe/patek-philippe-nautilus-tiffany...,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-226627.htm?caseMaterials=5&dialColor=702,/rolex/rolex-new-release-titanium-yacht-master...,/rolex/rolex-yacht-master-226627-42mm-titanium...,/rolex/rolex-2023-unworn-yacht-master-42mm-tit...,/rolex/yacht-master-226627-rlx-titanium-42mm-b...,/rolex/rolex-yacht-master-42-226627-titanium-n...,/rolex/rolex-yacht-master-42--id31167381.htm,/rolex/rolex-yacht-master-42---titanium---new-...,/rolex/rolex-yacht-master-42--id31430065.htm,/rolex/rolex-yacht-master-42--id30709557.htm,/rolex/yacht-master-42-42mm-titanium-ceramic-b...,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/seiko/ref-sbxb151.htm?caseMaterials=4,/seiko/seiko-seiko-astron-executive-line--id30...,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/ingersoll/ref-i09701.htm?caseMaterials=4&dialColor=708,/ingersoll/the-producer-blue--id27184420.htm,None,None,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/seiko/ref-ssqv042.htm?caseMaterials=5,/seiko/seiko-ssqv042---lukia-quartz-white-dial...,/seiko/----------ssqv042--id18656537.htm,/seiko/seiko-ssqv042---lukia-quartz-white-dial...,None,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
dbmodels[dbmodels.reset_index().groupby('url')[0].count()==2]

/tmp/ipykernel_43/1359073525.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dbmodels[dbmodels.reset_index().groupby('url')[0].count()==2]


,0,1,2,3,4,5,6,7,8,9,...,53,54,55,56,57,58,59,60,61,62
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/armandnicolet/ref-a486bgnnrma4480aa.htm?caseMaterials=4&dialColor=702,/armandnicolet/js9-gmt--id30674434.htm,/armandnicolet/armand-nicolet-js9-gmt--id19494...,/armandnicolet/armand-nicolet-js9-gmt--id25839...,/armandnicolet/js9-44-gmt-black-stainless-stee...,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/armandnicolet/ref-a486bgnnrma4480aa.htm?caseMaterials=4&dialColor=702,/armandnicolet/js9-gmt--id30674434.htm,/armandnicolet/armand-nicolet-js9-gmt--id19494...,/armandnicolet/armand-nicolet-js9-gmt--id25839...,/armandnicolet/js9-44-gmt-black-stainless-stee...,None,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
dbmodels.reset_index().url.nunique()

28139

In [ ]:
dict_models=dbmodels.drop_duplicates().apply(list,axis=1).to_dict()
#dict_models

In [ ]:
dbmod0=dbmod.loc[dbmodels.drop_duplicates().index].drop_duplicates()

In [ ]:
dbmod0.isna().sum()

brand                0
name                 0
refnum               0
modelimageurl        0
listings             0
                 ...  
58               27464
59               27477
60               27883
61               27969
62               27993
Length: 68, dtype: int64

In [ ]:
(dbmod0.groupby('refnum')['name'].count()>1).sum()

119

In [ ]:
dbmod0.loc[:,'name']=[(re.sub(r'\W+',' ',x)).lower() for x in list(dbmod0['name'])]

In [ ]:
dbmod0[dbmod0.brand==BRND].name.nunique()

647

In [ ]:
dbmod0.name.unique()

array(['sky moon tourbillon', '1908', 'nautilus', ...,
       'seiko astron executive line', 'the producer',
       'lukia solar automatic'], dtype=object)

In [ ]:
dbmod0[['name','refnum']][dbmod0[['name','refnum']].duplicated(keep=False).replace({True:1})==1]

,name,refnum
url,,
https://www.chrono24.com/mbf/ref-mad1.htm?caseMaterials=4&dialColor=704,1 edition,mad1
https://www.chrono24.com/grandseiko/ref-56467010.htm,grand seiko,56467010
https://www.chrono24.com/tagheuer/ref-cs2111.htm?caseMaterials=4&dialColor=702,monaco,cs2111
https://www.chrono24.com/madeditions/ref-mad1.htm?caseMaterials=4&dialColor=704,1 edition,mad1
https://www.chrono24.com/breguet/ref-3337.htm?dialColor=708,3337,3337
https://www.chrono24.com/heuer/ref-cs2111.htm?caseMaterials=4&dialColor=702,monaco,cs2111
https://www.chrono24.com/seiko/ref-56467010.htm?caseMaterials=4,grand seiko,56467010
https://www.chrono24.com/luminox/ref-3337.htm?caseMaterials=17,3337,3337


In [ ]:
dbmod0[dbmod0[['name','refnum']].duplicated()]

,brand,name,refnum,modelimageurl,listings,0,1,2,3,4,...,53,54,55,56,57,58,59,60,61,62
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/madeditions/ref-mad1.htm?caseMaterials=4&dialColor=704,M.A.D. Editions,1 edition,mad1,https://drive.google.com/open?id=1ND0h3jttEXaT...,33,/madeditions/mad-1--id28357361.htm,/madeditions/mad1-red-edition--id31156896.htm,/madeditions/mad-editions-mad1-edition-new-ful...,/madeditions/green--id31797868.htm,/madeditions/mad-edition-1--id30128897.htm,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/heuer/ref-cs2111.htm?caseMaterials=4&dialColor=702,Heuer,monaco,cs2111,https://drive.google.com/open?id=1v0e2XzlHdebN...,5,/heuer/heuer-monaco--id31114157.htm,/heuer/monaco-vintage-automatic--id29759866.htm,/heuer/cs2111--id31480732.htm,/heuer/monaco-chrono-limited-edition-cs2111-pe...,/heuer/heuer-monaco-cs2111--id29415641.htm,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/seiko/ref-56467010.htm?caseMaterials=4,Seiko,grand seiko,56467010,https://drive.google.com/open?id=11drjQTPS3rGl...,10,/seiko/1970s-grand-seiko-antique-5646-7010-56g...,/seiko/grand-seiko-gs-hi-beat-5646-7010-gs56--...,/seiko/gs--id29953416.htm,/seiko/grand-seiko-gs56-hi-beat-vws-2241--id30...,/seiko/5646-7010----cal5646a---739260ev20--id3...,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/luminox/ref-3337.htm?caseMaterials=17,Luminox,3337,3337,https://drive.google.com/open?id=10uk3gOLSZ0uO...,3,/luminox/luminox-command-raider-3320-series-re...,/luminox/luminox-commando-raider--id29118543.htm,/luminox/luminox-command-raider-3320-series-re...,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
dbmod0[dbmod0[['name','refnum']].duplicated()].index

Index(['https://www.chrono24.com/madeditions/ref-mad1.htm?caseMaterials=4&dialColor=704',
       'https://www.chrono24.com/heuer/ref-cs2111.htm?caseMaterials=4&dialColor=702',
       'https://www.chrono24.com/seiko/ref-56467010.htm?caseMaterials=4',
       'https://www.chrono24.com/luminox/ref-3337.htm?caseMaterials=17'],
      dtype='object', name='url')

In [ ]:
dbmod0.drop(list(dbmod0[dbmod0[['name','refnum']].duplicated()].index),axis=0,inplace=True)

In [ ]:
dbmod0[['name','refnum']].duplicated().sum()

0

In [ ]:
dbmod0

,brand,name,refnum,modelimageurl,listings,0,1,2,3,4,...,53,54,55,56,57,58,59,60,61,62
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/patekphilippe/ref-6002r001.htm?caseMaterials=1&dialColor=723,Patek Philippe,sky moon tourbillon,6002r001,https://drive.google.com/open?id=1TCb_TpAsqXxR...,6,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-52509.htm?caseMaterials=8,Rolex,1908,52509,https://drive.google.com/open?id=1c5unrOfEj0Bk...,3,/rolex/rolex-new-release-1908-white-gold-black...,/rolex/1908--id31783339.htm,/rolex/rolex-1908-perpetual-18k-39mm-white-gol...,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-52508.htm?caseMaterials=3,Rolex,1908,52508,https://drive.google.com/open?id=1_qq6lNga7E_5...,3,/rolex/rolex-new-release-1908-watch-52508--id3...,/rolex/rolex-1908-gelbgold--2023--neu--full-se...,/rolex/rolex-new-release-1908-watch-52508-blac...,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/patekphilippe/ref-57111a018.htm?caseMaterials=4&dialColor=710,Patek Philippe,nautilus,57111a018,https://drive.google.com/open?id=13N4xLm2iX3Bl...,12,/patekphilippe/patek-philippe-nautilus--id3021...,/patekphilippe/patek-philippe-nautilus-tiffany...,/patekphilippe/patek-philippe-nautilus--id2692...,/patekphilippe/nautilus-tiffany-and-co-blue-di...,/patekphilippe/nautilus-tiffany-amp-co-dial--s...,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-226627.htm?caseMaterials=5&dialColor=702,Rolex,yacht master 42,226627,https://drive.google.com/open?id=1gcO8Pq6AMYkJ...,16,/rolex/rolex-new-release-titanium-yacht-master...,/rolex/rolex-yacht-master-226627-42mm-titanium...,/rolex/rolex-2023-unworn-yacht-master-42mm-tit...,/rolex/yacht-master-226627-rlx-titanium-42mm-b...,/rolex/rolex-yacht-master-42-226627-titanium-n...,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/ref-l43594126.htm?caseMaterials=4&dialColor=701,Longines,longines lyre,l43594126,https://drive.google.com/open?id=11vs_h-5xeGuC...,4,/longines/longines-l43594126---longines-lyre-q...,/longines/longines-longines-lyre-watch-l435941...,/longines/longines-l43594126---longines-lyre-q...,/longines/longines-longines-watch-l43594126--i...,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/seiko/ref-sbxb151.htm?caseMaterials=4,Seiko,seiko astron executive line,sbxb151,https://drive.google.com/open?id=1Wq0rjkqpWB6T...,1,/seiko/seiko-seiko-astron-executive-line--id30...,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/ingersoll/ref-i09701.htm?caseMaterials=4&dialColor=708,Ingersoll,the producer,i09701,https://drive.google.com/open?id=1HmLfI_C7ytau...,1,/ingersoll/the-producer-blue--id27184420.htm,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
dbmod0=dbmod0.reset_index().set_index('refnum')
dbmod0=dbmod0[dbmod0.reset_index().groupby('refnum')['name'].count()==1]
dbmod0=dbmod0.reset_index()
dbmod0=dbmod0.set_index('url')

/tmp/ipykernel_43/3445905801.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dbmod0=dbmod0[dbmod0.reset_index().groupby('refnum')['name'].count()==1]


In [ ]:
dbmod0.refnum.nunique()

27758

In [ ]:
dbmod0

,refnum,brand,name,modelimageurl,listings,0,1,2,3,4,...,53,54,55,56,57,58,59,60,61,62
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/patekphilippe/ref-6002r001.htm?caseMaterials=1&dialColor=723,6002r001,Patek Philippe,sky moon tourbillon,https://drive.google.com/open?id=1TCb_TpAsqXxR...,6,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,/patekphilippe/patek-philippe-sky-moon-tourbil...,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-52509.htm?caseMaterials=8,52509,Rolex,1908,https://drive.google.com/open?id=1c5unrOfEj0Bk...,3,/rolex/rolex-new-release-1908-white-gold-black...,/rolex/1908--id31783339.htm,/rolex/rolex-1908-perpetual-18k-39mm-white-gol...,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-52508.htm?caseMaterials=3,52508,Rolex,1908,https://drive.google.com/open?id=1_qq6lNga7E_5...,3,/rolex/rolex-new-release-1908-watch-52508--id3...,/rolex/rolex-1908-gelbgold--2023--neu--full-se...,/rolex/rolex-new-release-1908-watch-52508-blac...,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/patekphilippe/ref-57111a018.htm?caseMaterials=4&dialColor=710,57111a018,Patek Philippe,nautilus,https://drive.google.com/open?id=13N4xLm2iX3Bl...,12,/patekphilippe/patek-philippe-nautilus--id3021...,/patekphilippe/patek-philippe-nautilus-tiffany...,/patekphilippe/patek-philippe-nautilus--id2692...,/patekphilippe/nautilus-tiffany-and-co-blue-di...,/patekphilippe/nautilus-tiffany-amp-co-dial--s...,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/rolex/ref-226627.htm?caseMaterials=5&dialColor=702,226627,Rolex,yacht master 42,https://drive.google.com/open?id=1gcO8Pq6AMYkJ...,16,/rolex/rolex-new-release-titanium-yacht-master...,/rolex/rolex-yacht-master-226627-42mm-titanium...,/rolex/rolex-2023-unworn-yacht-master-42mm-tit...,/rolex/yacht-master-226627-rlx-titanium-42mm-b...,/rolex/rolex-yacht-master-42-226627-titanium-n...,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/ref-l43594126.htm?caseMaterials=4&dialColor=701,l43594126,Longines,longines lyre,https://drive.google.com/open?id=11vs_h-5xeGuC...,4,/longines/longines-l43594126---longines-lyre-q...,/longines/longines-longines-lyre-watch-l435941...,/longines/longines-l43594126---longines-lyre-q...,/longines/longines-longines-watch-l43594126--i...,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/seiko/ref-sbxb151.htm?caseMaterials=4,sbxb151,Seiko,seiko astron executive line,https://drive.google.com/open?id=1Wq0rjkqpWB6T...,1,/seiko/seiko-seiko-astron-executive-line--id30...,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
https://www.chrono24.com/ingersoll/ref-i09701.htm?caseMaterials=4&dialColor=708,i09701,Ingersoll,the producer,https://drive.google.com/open?id=1HmLfI_C7ytau...,1,/ingersoll/the-producer-blue--id27184420.htm,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [ ]:
dict_hiercode_models={}
for bb in list(dbmod0.index):
  row=dbmod0.loc[bb,:]
  dict_hiercode_models[row.refnum]=row['brand']+"/" +row['name']+"/"+row['refnum']


In [ ]:
row

refnum                                                    5115j001
brand                                               Patek Philippe
name                                                     calatrava
modelimageurl    https://drive.google.com/open?id=1EaccnuRZ_iHS...
listings                                                         3
                                       ...                        
58                                                            None
59                                                            None
60                                                            None
61                                                            None
62                                                            None
Name: https://www.chrono24.com/patekphilippe/ref-5115j001.htm?caseMaterials=3&dialColor=701, Length: 68, dtype: object

In [ ]:
dict_hiercode_models

{'6002r001': 'Patek Philippe/sky moon tourbillon/6002r001',
 '52509': 'Rolex/1908/52509',
 '52508': 'Rolex/1908/52508',
 '57111a018': 'Patek Philippe/nautilus/57111a018',
 '226627': 'Rolex/yacht master 42/226627',
 '126529ln': 'Rolex/daytona/126529ln',
 'rm2703': 'Richard Mille/rafael nadal tourbillon/rm2703',
 'iw328903': 'IWC/ingenieur automatic/iw328903',
 '116595rbow': 'Rolex/daytona rainbow/116595rbow',
 '126500ln': 'Rolex/daytona/126500ln',
 'snxs79k1': 'Seiko/5 automatic sport snxs79/snxs79k1',
 '15417orzz1267or01a': 'Audemars Piguet/15417or zz 1267or 01 a/15417orzz1267or01a',
 'rm5201': 'Richard Mille/skull tourbillon/rm5201',
 '5531r012': 'Patek Philippe/world time minute repeater grand complications/5531r012',
 'sbdc091': 'Seiko/prospex alpinist/sbdc091',
 '887901': 'Cartier/santos/887901',
 '645003': 'Tutima/m2 chronograph/645003',
 'car221a': 'TAG Heuer/calibre 18 carrera/car221a',
 '26240baoo1320ba01': 'Audemars Piguet/royal oak selfwinding chronograph limited edition/2624

# Listings processing and rebuilding

In [ ]:
dbomega_listin=pd.read_json('/kaggle/input/listings1/listings')
dbomega_listinmod=dbomega_listin['Basic Info'].apply(pd.Series)
_IDX_=(dbomega_listinmod[dbomega_listinmod.Brand==BRND]).index
dbomega_listings=dbomega_listin.loc[_IDX_]

In [ ]:
BRND=re.sub('& ','',BRND)
BRND=re.sub('ö','',BRND)
BRND=re.sub('è','',BRND)
BRND=re.sub('ü','',BRND)

In [ ]:
dbomega_listings.refnum.nunique()

1405

In [ ]:
dbomega_listings.columns

Index(['URL', 'refnum', 'Basic Info', 'Caliber', 'Case', 'Bracelet/strap',
       'Description', 'Seller INFO', 'IMG', 'Functions', 'Other'],
      dtype='object')

In [ ]:
dbomegacol=dbomega_listings.columns

In [ ]:
dbomega_listings['Basic Info'].apply(pd.Series)

,Listing code,Brand,Model,Reference number,Dealer product code,Movement,Case material,Bracelet material,Year of production,Condition,Scope of delivery,Gender,Location,Price,Availability
702,IHS0Z5,Longines,HydroConquest,L3.790.4.96.6,8694716006724,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Germany, Bamberg","€2,440 (= $2,702)",Item needs to be procured
928,HNRNU7,Longines,Conquest Heritage,L1.641.4,2300207,Automatic,Steel,NaN,Unknown,Good (Light signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"Japan, NAGOYA","¥187,000 (= $1,320)",Item needs to be procured
1148,IQBR25,Longines,Conquest,L3.835.4.72.6,23326,NaN,NaN,NaN,2023,"New (Brand new, without any signs of wear)","Original box, original papers",NaN,"Turkey, Istanbul","€2,991 (= $3,312)",Item is in stock
1579,H9IK28,Longines,Legend Diver,L3.674.4,NaN,Automatic,Steel,Leather,2017,Very good (Worn with little to no signs of wear),"Original box, original papers",Men's watch/Unisex,"Greece, Athens","€1,785 (= $1,977)",Item needs to be procured
1794,I5NQM5,Longines,Conquest,L3.830.4.02.6,NaN,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"United States of America, California, Los Angeles","$1,699 [Negotiable]",Item is in stock
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249166,IVOBE1,Longines,NaN,L37284666,NaN,NaN,Steel,NaN,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",NaN,"Japan, TAKASAKISHI-SHI","¥128,300 (= $898)",Item needs to be procured
249167,F7U7X5,Longines,Record,L23214566,NaN,Automatic,Steel,Steel,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Women's watch,"Japan, SAITAMASHI","¥179,800 (= $1,258)",Item needs to be procured
249170,I137H2,Longines,DolceVita,L57554576,NaN,NaN,Steel,NaN,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Japan, Chiba-ken","¥109,800 (= $768) [Negotiable]",Item needs to be procured
249173,IGU8W1,Longines,Flagship,L4.899.3.92.7,NaN,NaN,NaN,NaN,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Japan, SAITAMASHI","¥179,800 (= $1,258)",Item available on request


In [ ]:
dbomegalist_basic_info=dbomega_listings['Basic Info'].apply(pd.Series)

In [ ]:
try:
  dbomegalist_basic_info.Model.nunique()
except:
    _=0

In [ ]:
dbomegalist_basic_info.columns

Index(['Listing code', 'Brand', 'Model', 'Reference number',
       'Dealer product code', 'Movement', 'Case material', 'Bracelet material',
       'Year of production', 'Condition', 'Scope of delivery', 'Gender',
       'Location', 'Price', 'Availability'],
      dtype='object')

In [ ]:
#dbomega_listings['Caliber'].apply(pd.Series)[500:550]

In [ ]:
#dbomega_listings['Caliber'].apply(pd.Series)[0].isna().sum()

In [ ]:
dbomega_listings['Case'].apply(pd.Series)

,Case material,Case diameter,Water resistance,Crystal,Dial,Thickness,0,Bezel material,Dial numerals
702,Steel,41 mm Try it on,30 ATM,Sapphire crystal,Blue,NaN,NaN,NaN,NaN
928,Steel,38.5 mm Try it on,NaN,Plastic,Silver,12.5 mm,NaN,NaN,NaN
1148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1579,Steel,42 mm Try it on,NaN,NaN,Black,NaN,NaN,NaN,NaN
1794,Steel,41 mm Try it on,10 ATM,Sapphire crystal,Green,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
249166,Steel,NaN,NaN,NaN,Black,NaN,NaN,NaN,NaN
249167,Steel,NaN,3 ATM,Sapphire crystal,Black,NaN,NaN,NaN,NaN
249170,Steel,NaN,NaN,NaN,Black,NaN,NaN,NaN,NaN
249173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#dbomega_listings['Case'].apply(pd.Series)[0].isna().sum()

In [ ]:
dbomega_listings['Bracelet/strap'].apply(pd.Series)

,Bracelet material,Bracelet color,Clasp,0,Clasp material,Bracelet length,Lug width,Buckle width,Bracelet thickness
702,Steel,Steel,Fold clasp,NaN,NaN,NaN,NaN,NaN,NaN
928,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1148,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1579,Leather,Black,Buckle,NaN,Steel,NaN,NaN,NaN,NaN
1794,Steel,Steel,Fold clasp,NaN,Steel,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
249166,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
249167,Steel,Steel,Fold clasp,NaN,Steel,NaN,NaN,NaN,NaN
249170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
249173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#dbomega_listings['Bracelet/strap'].apply(pd.Series)[0].isna().sum()

In [ ]:
dbomega_listings['Other'].apply(pd.Series)

,data,0
702,"Luminous hands, Rotating Bezel, Luminous indices",NaN
928,NaN,NaN
1148,NaN,NaN
1579,Luminous hands,NaN
1794,NaN,NaN
...,...,...
249166,NaN,NaN
249167,NaN,NaN
249170,NaN,NaN
249173,NaN,NaN


In [ ]:
#dbomega_listings['Other'].apply(pd.Series)[0].isna().sum()

In [ ]:
dbomega_listings['Description'].apply(pd.Series)

,data,0
702,"This model is original, new, unworn and comes ...",NaN
928,[product code] 2300207 [Brand name] LONGINES [...,NaN
1148,NaN,NaN
1579,NaN,NaN
1794,Item Longines Men's Watch Model # L3.830.4.02....,NaN
...,...,...
249166,ロンジン - L3.728.4.66.6 コンクエスト V.H.P. GMT 43 ステンレ...,NaN
249167,brand Longines,NaN
249170,NaN,NaN
249173,Strap length [arm circumference] approximately...,NaN


In [ ]:
#dbomega_listings['Description'].apply(pd.Series)[0].isna().sum()

In [ ]:
dbomega_listings['IMG'].apply(pd.Series)

,zoom
702,[https://cdn2.chrono24.com/images/uhren/310625...
928,[]
1148,[]
1579,[https://cdn2.chrono24.com/images/uhren/289974...
1794,[]
...,...
249166,[]
249167,[]
249170,[]
249173,[]


In [ ]:
dbomega_listings['Functions'].apply(pd.Series)

,data,0
702,"Date, GMT",NaN
928,Chronograph,NaN
1148,NaN,NaN
1579,Date,NaN
1794,NaN,NaN
...,...,...
249166,NaN,NaN
249167,Date,NaN
249170,NaN,NaN
249173,NaN,NaN


In [ ]:
#dbomega_listings['Functions'].apply(pd.Series)[0].isna().sum()

In [ ]:
dbomega_listings['Seller INFO'].apply(pd.Series)

,Seller,Seller rating,SellerID
702,Horando Deutschland GmbH,"4.4 Reviews (1,672)",31062563
928,PAWNSHOP ITO,4.8 Reviews (107),29662410
1148,Horomania,4.6 Reviews (189),31461374
1579,Chronos-Ora24,4.9 Reviews (542),28997426
1794,Incmaxx,4.9 Reviews (186),30497134
...,...,...,...
249166,IPPO JAPAN WATCH,"4.9 Reviews (5,381)",31710938
249167,Hachimura Watch Japan,"4.9 Reviews (3,519)",25559997
249170,TIMESMAN CO LTD,4.9 Reviews (393),30283901
249173,Hachimura Watch Japan,"4.9 Reviews (3,519)",31018784


In [ ]:
dbomega_listings_exp=dbomega_listings['URL'].apply(pd.Series)
dbomega_listings_exp=pd.concat([dbomega_listings_exp,dbomega_listings[['refnum']]],axis=1)
for  dt in dbomega_listings.columns[2::]:
  dbomega_listings_exp=pd.concat([dbomega_listings_exp,dbomega_listings[dt].apply(pd.Series).drop([0],axis=1,errors='ignore')],axis=1)


In [ ]:
list_control=['Reference number','Model','Year of production','Caliber/movement','Base caliber','Price','Dial','Bracelet material','Bezel material','Dial numerals','Bracelet color','Clasp material','Clasp','Case material']
for q in list_control:
    if q not in list(dbomega_listings_exp.columns):
        sys.exit(f'critical column:{q} not in dataset,brand is not processed')

In [ ]:
dbomega_listings_exp.Model.unique()

array(['HydroConquest', 'Conquest Heritage', 'Conquest', 'Legend Diver',
       'Master Collection', nan, 'PrimaLuna', 'Heritage', 'Spirit',
       'Record', 'Elegant', 'La Grande Classique', 'Oposition', 'Lyre',
       'DolceVita', 'Lindbergh Hour Angle', 'Conquest Classic',
       'Avigation', 'Saint-Imier', 'Flagship', 'Présence', 'Equestrian',
       'Symphonette', 'Evidenza', 'Admiral', 'Column-Wheel Chronograph',
       'Twenty-Four Hours', 'Flagship Heritage', 'Grande Vitesse',
       'Présence Heritage'], dtype=object)

In [ ]:
dbomega_listings_exp[100:200]

,url,refnum,Listing code,Brand,Model,Reference number,Dealer product code,Movement,Case material,Bracelet material,...,Lug width,Buckle width,Bracelet thickness,data,Seller,Seller rating,SellerID,zoom,data,data
20315,https://www.chrono24.com/longines/longines-hyd...,l37404566,EGYY32,Longines,HydroConquest,L3.740.4.56.6,NaN,Quartz,Steel,Steel,...,NaN,NaN,NaN,Welcome to our shop! All our watches sold on C...,EW WATCHES,"4.8 Reviews (1,325)",24306411,[https://cdn2.chrono24.com/cdn-cgi/image/f=aut...,Date,Central seconds
20363,https://www.chrono24.com/longines/longines-hyd...,l37844569,BAU1C3,Longines,HydroConquest,L3.784.4.56.9,L3.784.4.56.9,Automatic,Ceramic,Rubber,...,NaN,NaN,NaN,NaN,Watch Philosophy,4.3 Reviews (503),18981264,[],NaN,NaN
20377,https://www.chrono24.com/longines/spirit--id30...,l38104936,I2EGO1,Longines,Spirit,L38104936,NaN,Automatic,Steel,Steel,...,NaN,NaN,NaN,Watch never worn. Under warranty filmed as pic...,Private Seller,,30345144,[https://cdn2.chrono24.com/cdn-cgi/image/f=aut...,NaN,Chronometer
20382,https://www.chrono24.com/longines/longines-lon...,l38104936,IK4UE6,Longines,Spirit,L38104936,NaN,Automatic,Steel,Steel,...,NaN,NaN,NaN,- Delivery time: In stock. Available within 5-...,WA Watch Atelier e.K.,4.9 Reviews (69),31172486,[],NaN,Chronometer
20499,https://www.chrono24.com/longines/--l42594126-...,l42594126,IQA5C5,Longines,Lyre,L4.259.4.12.6,270-003-807-7835,Quartz,Steel,Steel,...,NaN,NaN,NaN,ケース径約: 25mm 腕周り最大約: 17cm,"Komehyo Co.,Ltd.",4.8 Reviews (925),31459296,[https://cdn2.chrono24.com/images/uhren/314592...,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23659,https://www.chrono24.com/longines/longines-l28...,l28224569,7RTGN0,Longines,NaN,L28224569,13120A,Automatic,Steel,Rubber,...,NaN,NaN,NaN,【Product Information】 Brand: Longines Ref. No....,The Watch Company,"4.5 Reviews (2,742)",13055207,[],NaN,NaN
23703,https://www.chrono24.com/longines/longines-aut...,l38414966,H3J8S3,Longines,HydroConquest,L3.841.4.96.6,18,Automatic,Steel,Steel,...,NaN,NaN,NaN,NaN,Fabel-Time GmbH,"4.9 Reviews (2,565)",28718380,[https://cdn2.chrono24.com/cdn-cgi/image/f=aut...,Date,"Central seconds, Luminous hands, Luminous indices"
23704,https://www.chrono24.com/longines/-longines---...,l27414732,IDRZL4,Longines,NaN,L2.741.4.73.2,H2660,Automatic,Steel,NaN,...,19 mm Size guide,19 mm,NaN,-Product description- Details: LONGINES Herita...,Jewel & Watch Supply,4.8 Reviews (209),30875889,[],Chronograph,NaN
23749,https://www.chrono24.com/longines/longines-lon...,l38104936,IDLH65,Longines,Spirit,L38104936,NaN,Automatic,Steel,Steel,...,NaN,NaN,NaN,NaN,Time Trader,4.9 Reviews (749),30867450,[],NaN,Chronometer


In [ ]:
dbomega_listings_exp.isna().sum()

url                       0
refnum                    0
Listing code              0
Brand                     0
Model                   517
Reference number         18
Dealer product code    3844
Movement               1036
Case material           959
Bracelet material      1706
Year of production        0
Condition                34
Scope of delivery         0
Gender                 1145
Location                  0
Price                    25
Availability             25
Movement               1036
Caliber/movement       7672
Power reserve          7505
Number of jewels       9369
Frequency              9782
Base caliber           9708
Case material           959
Case diameter          1920
Water resistance       4335
Crystal                4149
Dial                   1720
Thickness              8350
Bezel material         7662
Dial numerals          7663
Bracelet material      1706
Bracelet color         4512
Clasp                  5871
Clasp material         6764
Bracelet length     

In [ ]:
dbomega_listings_exp=dbomega_listings_exp.fillna('Unknown')

In [ ]:
lst_ind=list(dbomega_listings_exp.columns)

In [ ]:
dic_ind0={}
dic_ind={}
other=list(dbomegacol).index('Other')
description=list(dbomegacol).index('Description')
functions=list(dbomegacol).index('Functions')
lst_ind0=[other,description,functions]
dic_ind0['1']=other
dic_ind0['2']=description
dic_ind0['3']=functions
if dic_ind0['1']==max(lst_ind0):
    dic_ind['1']=3
elif dic_ind0['1']==min(lst_ind0):
    dic_ind['1']=1
else:
    dic_ind['1']=2
if dic_ind0['2']==max(lst_ind0):
    dic_ind['2']=3
elif dic_ind0['2']==min(lst_ind0):
    dic_ind['2']=1
else:
    dic_ind['2']=2
if dic_ind0['3']==max(lst_ind0):
    dic_ind['3']=3
elif dic_ind0['3']==min(lst_ind0):
    dic_ind['3']=1
else:
    dic_ind['3']=2
dic_ind

{'1': 3, '2': 1, '3': 2}

In [ ]:
cct0=0
cct=0
for vv in lst_ind:
  if vv=='data':
   cct=cct+1
   for key, value in dic_ind.items():
     if value == cct:
      lst_ind[cct0]='data'+key
  cct0=cct0+1

In [ ]:
pd.Index(lst_ind)

Index(['url', 'refnum', 'Listing code', 'Brand', 'Model', 'Reference number',
       'Dealer product code', 'Movement', 'Case material', 'Bracelet material',
       'Year of production', 'Condition', 'Scope of delivery', 'Gender',
       'Location', 'Price', 'Availability', 'Movement', 'Caliber/movement',
       'Power reserve', 'Number of jewels', 'Frequency', 'Base caliber',
       'Case material', 'Case diameter', 'Water resistance', 'Crystal', 'Dial',
       'Thickness', 'Bezel material', 'Dial numerals', 'Bracelet material',
       'Bracelet color', 'Clasp', 'Clasp material', 'Bracelet length',
       'Lug width', 'Buckle width', 'Bracelet thickness', 'data2', 'Seller',
       'Seller rating', 'SellerID', 'zoom', 'data3', 'data1'],
      dtype='object')

In [ ]:
dbomega_listings_exp=pd.DataFrame(np.array(dbomega_listings_exp),columns=pd.Index(lst_ind)).drop(['zoom'],axis=1,errors='ignore')

In [ ]:
dbomega_listings_exp

,url,refnum,Listing code,Brand,Model,Reference number,Dealer product code,Movement,Case material,Bracelet material,...,Bracelet length,Lug width,Buckle width,Bracelet thickness,data2,Seller,Seller rating,SellerID,data3,data1
0,https://www.chrono24.com/longines/hydroconques...,l37904966,IHS0Z5,Longines,HydroConquest,L3.790.4.96.6,8694716006724,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,"This model is original, new, unworn and comes ...",Horando Deutschland GmbH,"4.4 Reviews (1,672)",31062563,"Date, GMT","Luminous hands, Rotating Bezel, Luminous indices"
1,https://www.chrono24.com/longines/-----l16414-...,l16414,HNRNU7,Longines,Conquest Heritage,L1.641.4,2300207,Automatic,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,[product code] 2300207 [Brand name] LONGINES [...,PAWNSHOP ITO,4.8 Reviews (107),29662410,Chronograph,Unknown
2,https://www.chrono24.com/longines/conquest-l38...,l38354726,IQBR25,Longines,Conquest,L3.835.4.72.6,23326,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Horomania,4.6 Reviews (189),31461374,Unknown,Unknown
3,https://www.chrono24.com/longines/longines-leg...,l36744,H9IK28,Longines,Legend Diver,L3.674.4,Unknown,Automatic,Steel,Leather,...,Unknown,Unknown,Unknown,Unknown,Unknown,Chronos-Ora24,4.9 Reviews (542),28997426,Date,Luminous hands
4,https://www.chrono24.com/longines/longines-con...,l38304026,I5NQM5,Longines,Conquest,L3.830.4.02.6,Unknown,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,Item Longines Men's Watch Model # L3.830.4.02....,Incmaxx,4.9 Reviews (186),30497134,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9924,https://www.chrono24.com/longines/longines-lon...,l37284666,IVOBE1,Longines,Unknown,L37284666,Unknown,Unknown,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,ロンジン - L3.728.4.66.6 コンクエスト V.H.P. GMT 43 ステンレ...,IPPO JAPAN WATCH,"4.9 Reviews (5,381)",31710938,Unknown,Unknown
9925,https://www.chrono24.com/longines/longines-lon...,l23214566,F7U7X5,Longines,Record,L23214566,Unknown,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,brand Longines,Hachimura Watch Japan,"4.9 Reviews (3,519)",25559997,Date,Unknown
9926,https://www.chrono24.com/longines/longines-lon...,l57554576,I137H2,Longines,DolceVita,L57554576,Unknown,Unknown,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,TIMESMAN CO LTD,4.9 Reviews (393),30283901,Unknown,Unknown
9927,https://www.chrono24.com/longines/longines-lon...,l48993927,IGU8W1,Longines,Flagship,L4.899.3.92.7,Unknown,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Strap length [arm circumference] approximately...,Hachimura Watch Japan,"4.9 Reviews (3,519)",31018784,Unknown,Unknown


In [ ]:
dbomega_listings_exp.groupby('Model')['refnum'].count()

Model
Admiral                       30
Avigation                     95
Column-Wheel Chronograph      15
Conquest                    1012
Conquest Classic             301
Conquest Heritage            105
DolceVita                    578
Elegant                      366
Equestrian                    33
Evidenza                     118
Flagship                     351
Flagship Heritage             63
Grande Vitesse                 9
Heritage                     383
HydroConquest               1143
La Grande Classique          712
Legend Diver                 234
Lindbergh Hour Angle          36
Lyre                         120
Master Collection           1664
Oposition                     12
PrimaLuna                    261
Présence                     377
Présence Heritage             16
Record                       506
Saint-Imier                   44
Spirit                       783
Symphonette                   36
Twenty-Four Hours              9
Unknown                      517
Name

In [ ]:
dbomega_listings_exp.groupby('Model')['refnum'].nunique()

Model
Admiral                       9
Avigation                     7
Column-Wheel Chronograph      2
Conquest                    194
Conquest Classic             47
Conquest Heritage            13
DolceVita                   107
Elegant                      74
Equestrian                    8
Evidenza                     20
Flagship                     71
Flagship Heritage             5
Grande Vitesse                2
Heritage                     58
HydroConquest               105
La Grande Classique         151
Legend Diver                 20
Lindbergh Hour Angle          7
Lyre                         24
Master Collection           207
Oposition                     3
PrimaLuna                    42
Présence                     64
Présence Heritage             2
Record                       83
Saint-Imier                  12
Spirit                       64
Symphonette                   8
Twenty-Four Hours             2
Unknown                     170
Name: refnum, dtype: int64

In [ ]:
(dbomega_listings_exp.groupby('url')['refnum'].nunique()==1).sum()

9929

In [ ]:
(dbomega_listings_exp['Price']=='Unknown').sum()

25

In [ ]:
dbomega_listings_exp=dbomega_listings_exp.set_index('url')

In [ ]:
dbomega_listings_exp

,refnum,Listing code,Brand,Model,Reference number,Dealer product code,Movement,Case material,Bracelet material,Year of production,...,Bracelet length,Lug width,Buckle width,Bracelet thickness,data2,Seller,Seller rating,SellerID,data3,data1
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/longines/hydroconquest-gmt--id31062563.htm,l37904966,IHS0Z5,Longines,HydroConquest,L3.790.4.96.6,8694716006724,Automatic,Steel,Steel,2023,...,Unknown,Unknown,Unknown,Unknown,"This model is original, new, unworn and comes ...",Horando Deutschland GmbH,"4.4 Reviews (1,672)",31062563,"Date, GMT","Luminous hands, Rotating Bezel, Luminous indices"
https://www.chrono24.com/longines/-----l16414-ss---2300207--id29662410.htm,l16414,HNRNU7,Longines,Conquest Heritage,L1.641.4,2300207,Automatic,Steel,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,[product code] 2300207 [Brand name] LONGINES [...,PAWNSHOP ITO,4.8 Reviews (107),29662410,Chronograph,Unknown
https://www.chrono24.com/longines/conquest-l38354726--id31461374.htm,l38354726,IQBR25,Longines,Conquest,L3.835.4.72.6,23326,Unknown,Unknown,Unknown,2023,...,Unknown,Unknown,Unknown,Unknown,Unknown,Horomania,4.6 Reviews (189),31461374,Unknown,Unknown
https://www.chrono24.com/longines/longines-legend-diver-42mm-automatic-date-2017-full-set--id28997426.htm,l36744,H9IK28,Longines,Legend Diver,L3.674.4,Unknown,Automatic,Steel,Leather,2017,...,Unknown,Unknown,Unknown,Unknown,Unknown,Chronos-Ora24,4.9 Reviews (542),28997426,Date,Luminous hands
https://www.chrono24.com/longines/longines-conquest-2023-41mm-green-dial-steel-mens-watch-l38304026--id30497134.htm,l38304026,I5NQM5,Longines,Conquest,L3.830.4.02.6,Unknown,Automatic,Steel,Steel,2023,...,Unknown,Unknown,Unknown,Unknown,Item Longines Men's Watch Model # L3.830.4.02....,Incmaxx,4.9 Reviews (186),30497134,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/longines-longines---l37284666-conquest-vhp-gmt-43-stainless-steel--carbon--bracelet-b--id31710938.htm,l37284666,IVOBE1,Longines,Unknown,L37284666,Unknown,Unknown,Steel,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,ロンジン - L3.728.4.66.6 コンクエスト V.H.P. GMT 43 ステンレ...,IPPO JAPAN WATCH,"4.9 Reviews (5,381)",31710938,Unknown,Unknown
https://www.chrono24.com/longines/longines-longines-record-watch-l23214566--id25559997.htm,l23214566,F7U7X5,Longines,Record,L23214566,Unknown,Automatic,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,brand Longines,Hachimura Watch Japan,"4.9 Reviews (3,519)",25559997,Date,Unknown
https://www.chrono24.com/longines/longines-longines-dolcevita-black-dial-mens-watch-l57554576--id30283901.htm,l57554576,I137H2,Longines,DolceVita,L57554576,Unknown,Unknown,Steel,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,TIMESMAN CO LTD,4.9 Reviews (393),30283901,Unknown,Unknown


In [ ]:
list(dbomega_listings_exp['Year of production'])

['2023',
 'Unknown',
 '2023',
 '2017',
 '2023',
 '2023',
 '1964',
 '2013',
 '2023',
 '2023 (Approximation)',
 '2023',
 '2023',
 '2023',
 '2015',
 '2021 (Approximation)',
 'Unknown',
 '2023',
 '2023',
 'Unknown',
 '2023',
 '2023',
 '2023',
 'Unknown',
 '2023',
 '2023',
 '2023',
 'Unknown',
 '2023',
 'Unknown',
 'Unknown',
 '2023',
 '2023 (Approximation)',
 '2023',
 '2023',
 '2023',
 'Unknown',
 '2023',
 '2023',
 '2023',
 '2023',
 '2016',
 '2023',
 '2023',
 '2023',
 '2023',
 '2016 (Approximation)',
 '2023',
 '2023',
 '2006',
 '2009',
 'Unknown',
 '2023',
 '2016',
 '2023',
 '2017',
 'Unknown',
 '2019',
 '2023 (Approximation)',
 'Unknown',
 '2015',
 'Unknown',
 '2023',
 '2023',
 '2023 (Approximation)',
 '2023',
 '2016',
 '2023',
 '2023',
 '2023',
 '2023',
 'Unknown',
 '2022',
 '2023',
 '2023',
 '2021',
 '2022',
 'Unknown',
 '2023',
 '2023',
 'Unknown',
 '2022',
 '2023',
 '2021',
 '2022',
 'Unknown',
 '2023',
 'Unknown',
 'Unknown',
 '2023',
 'Unknown',
 'Unknown',
 '2023',
 'Unknown',
 'Un

In [ ]:
if 'Number of jewels' in list(dbomega_listings_exp.columns):
    _=0
else:
   dbomega_listings_exp['Number of jewels']='Unknown'

In [ ]:
extn=list(dbomega_listings_exp.columns)[3:]
extn.remove('Model')
dbomega_listings_exp=dbomega_listings_exp[['refnum', 'Listing code', 'Brand','Model']+extn]

In [ ]:
dbomega_listings_exp.loc[:,'Model']=[(re.sub(r'\W+',' ',x)).lower() if x!='Unknown' else x for x in list(dbomega_listings_exp['Model'])]

In [ ]:
dbomelist_exp=dbomega_listings_exp.copy()

# Listings processing and rebuilding cont...

## Feature engineering

In [ ]:
dbomega_listings_exp.reset_index().url.nunique()

9929

In [ ]:
dbomega_test=dbomelist_exp[dbomelist_exp.reset_index().groupby('url')['refnum'].count()==2].sort_index()
dbomega_test

/tmp/ipykernel_43/846380479.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dbomega_test=dbomelist_exp[dbomelist_exp.reset_index().groupby('url')['refnum'].count()==2].sort_index()


,refnum,Listing code,Brand,Model,Reference number,Dealer product code,Movement,Movement,Case material,Case material,...,Bracelet length,Lug width,Buckle width,Bracelet thickness,data2,Seller,Seller rating,SellerID,data3,data1
url,,,,,,,,,,,,,,,,,,,,,


In [ ]:
dbomelist_exp=dbomelist_exp[dbomelist_exp.reset_index().groupby('url')['refnum'].count()==1]
dbomelist_exp

/tmp/ipykernel_43/2483546375.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  dbomelist_exp=dbomelist_exp[dbomelist_exp.reset_index().groupby('url')['refnum'].count()==1]


,refnum,Listing code,Brand,Model,Reference number,Dealer product code,Movement,Movement,Case material,Case material,...,Bracelet length,Lug width,Buckle width,Bracelet thickness,data2,Seller,Seller rating,SellerID,data3,data1
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/longines/hydroconquest-gmt--id31062563.htm,l37904966,IHS0Z5,Longines,hydroconquest,L3.790.4.96.6,8694716006724,Automatic,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,"This model is original, new, unworn and comes ...",Horando Deutschland GmbH,"4.4 Reviews (1,672)",31062563,"Date, GMT","Luminous hands, Rotating Bezel, Luminous indices"
https://www.chrono24.com/longines/-----l16414-ss---2300207--id29662410.htm,l16414,HNRNU7,Longines,conquest heritage,L1.641.4,2300207,Automatic,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,[product code] 2300207 [Brand name] LONGINES [...,PAWNSHOP ITO,4.8 Reviews (107),29662410,Chronograph,Unknown
https://www.chrono24.com/longines/conquest-l38354726--id31461374.htm,l38354726,IQBR25,Longines,conquest,L3.835.4.72.6,23326,Unknown,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Horomania,4.6 Reviews (189),31461374,Unknown,Unknown
https://www.chrono24.com/longines/longines-legend-diver-42mm-automatic-date-2017-full-set--id28997426.htm,l36744,H9IK28,Longines,legend diver,L3.674.4,Unknown,Automatic,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,Unknown,Chronos-Ora24,4.9 Reviews (542),28997426,Date,Luminous hands
https://www.chrono24.com/longines/longines-conquest-2023-41mm-green-dial-steel-mens-watch-l38304026--id30497134.htm,l38304026,I5NQM5,Longines,conquest,L3.830.4.02.6,Unknown,Automatic,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,Item Longines Men's Watch Model # L3.830.4.02....,Incmaxx,4.9 Reviews (186),30497134,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/longines-longines---l37284666-conquest-vhp-gmt-43-stainless-steel--carbon--bracelet-b--id31710938.htm,l37284666,IVOBE1,Longines,Unknown,L37284666,Unknown,Unknown,Unknown,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,ロンジン - L3.728.4.66.6 コンクエスト V.H.P. GMT 43 ステンレ...,IPPO JAPAN WATCH,"4.9 Reviews (5,381)",31710938,Unknown,Unknown
https://www.chrono24.com/longines/longines-longines-record-watch-l23214566--id25559997.htm,l23214566,F7U7X5,Longines,record,L23214566,Unknown,Automatic,Automatic,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,brand Longines,Hachimura Watch Japan,"4.9 Reviews (3,519)",25559997,Date,Unknown
https://www.chrono24.com/longines/longines-longines-dolcevita-black-dial-mens-watch-l57554576--id30283901.htm,l57554576,I137H2,Longines,dolcevita,L57554576,Unknown,Unknown,Unknown,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,Unknown,TIMESMAN CO LTD,4.9 Reviews (393),30283901,Unknown,Unknown


In [ ]:
dbomelist_exp.loc[:,'Year of production']=[re.search(r'\d+',x).group() if re.search(r'\d+',x) else x for x in list(dbomelist_exp['Year of production'])]

In [ ]:
dbomelist_exp['Caliber num digits']=''

In [ ]:
dbomelist_exp.loc[:,'Caliber num digits']=[re.search(r'\d+',x).group() if re.search(r'\d+',x) else x for x in list(dbomelist_exp['Caliber/movement'])]

In [ ]:
for hh in list(dbomelist_exp.index):
  if dbomelist_exp.loc[hh,'Base caliber']=='Unknown':
   if dbomelist_exp.loc[hh,'Caliber num digits']!='Unknown':
    dbomelist_exp.loc[hh,'Base caliber']=dbomelist_exp.loc[hh,'Brand']+' '+dbomelist_exp.loc[hh,'Caliber num digits']

In [ ]:
dbomelist_exp.drop(['Caliber num digits'],axis=1,inplace=True)

In [ ]:
for hh in list(dbomelist_exp.index):
  if dbomelist_exp.loc[hh,'Base caliber']!='Unknown':
    if re.search(r'\d{2,50}',dbomelist_exp.loc[hh,'Base caliber']):
       if  re.search(r'(\D{2,50})(\d{2,50})',dbomelist_exp.loc[hh,'Base caliber']):
         _=0
         #print('correct base caliber with high probability')
       elif  re.search(r'(\d{2,50})(\D{2,50})',dbomelist_exp.loc[hh,'Base caliber']):
        _=0
       elif dbomelist_exp.loc[hh,'Base caliber'].isdigit():
         dbomelist_exp.loc[hh,'Base caliber']=dbomelist_exp.loc[hh,'Brand']+' '+re.search(r'\d+',dbomelist_exp.loc[hh,'Base caliber']).group()
       else:
         print(dbomelist_exp.loc[hh,'Base caliber'])
         dbomelist_exp.drop(hh,axis=0,inplace=True)
    else:
       print(dbomelist_exp.loc[hh,'Base caliber'])
       dbomelist_exp.drop(hh,axis=0,inplace=True)

Longines Self-winding mechanical
Longines Quartz
L888
L687.2
Longines A
Automatic
L420
自動巻
Longines Q
ETA
自動巻メカニカルムーブメント
ETA
Longines Automatyczny
Longines Kwarcowy
Longines Quartz
L888
Longines Eigenkonstruktion auf Basis ETA
Longines Automatic
L888
Longines Automatic
L667
自動巻メカニカルムーブメント
Longines A
Longines Q
L888
Longines CUARZO
クォーツ
クォーツ
L420
Longines Eta
自動巻メカニカルムーブメント
クォーツ
L619
自動巻メカニカルムーブメント
Longines Q
Longines A
自動巻
Longines 3
L888.4
Automatic
Longines A
Longines kwarcowy
L895
Longines A
L791
自動巻
Longines SELF WINDING
Longines automatyczny
Longines AUTOMATIC
ETA
ETA
L888.2
自動巻
L888
自動巻メカニカルムーブメント
Automatico
12.68Z
Longines A
Longines AUTOMATIC
ETA
L888.4
L888.4
L895
Longines QUARTZ
Longines Automatic
L888.4
自動巻メカニカルムーブメント
L888
Longines Q
L688
L791.4
自動巻
Longines A
L888
ETA
L836
Longines A
自動巻
Longines QUARTZ
Longines A
Longines 4
自動巻
Automatic
自動巻
自動巻メカニカルムーブメント
L899
Automatic
L888
自動巻メカニカルムーブメント
Quartz
Longines A
Longines
Longines A
Automatic
30L
Longines Quartz
1268z
L844.4
Lo

In [ ]:
for hh in list(dbomelist_exp.index):
  if  re.search('cal',dbomelist_exp.loc[hh,'Base caliber'].lower()):
         print(dbomelist_exp.loc[hh,'Base caliber'])
         dbomelist_exp.loc[hh,'Base caliber']=dbomelist_exp.loc[hh,'Brand']+' '+re.search(r'\d+',dbomelist_exp.loc[hh,'Base caliber']).group()

Calibro L592
Longines Caliber L688
Cal.876
Caliber 15.68Z
Longines Calibre L250
Caliber L667


In [ ]:
for hh in list(dbomelist_exp.index):
  if  re.search('rolex',dbomelist_exp.loc[hh,'Base caliber'].lower()):
         print(dbomelist_exp.loc[hh,'Base caliber'])
         dbomelist_exp.drop(hh,axis=0,inplace=True)

In [ ]:
#dbomelist_exp[['Base caliber']][100:150]

In [ ]:
dbomelist_exp.loc[:,'Seller rating']=[re.search(r'\d+\.?\d+',x).group() if re.search(r'\d+\.?\d+',x) else x for x in list(dbomelist_exp['Seller rating'])]

In [ ]:
dbomelist_exp=dbomelist_exp.drop(['Listing code','Seller','SellerID'],axis=1)

In [ ]:
dbomelist_exp

,refnum,Brand,Model,Reference number,Dealer product code,Movement,Movement,Case material,Case material,Bracelet material,...,Clasp,Clasp material,Bracelet length,Lug width,Buckle width,Bracelet thickness,data2,Seller rating,data3,data1
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/longines/hydroconquest-gmt--id31062563.htm,l37904966,Longines,hydroconquest,L3.790.4.96.6,8694716006724,Automatic,Automatic,Steel,Steel,Steel,...,Fold clasp,Unknown,Unknown,Unknown,Unknown,Unknown,"This model is original, new, unworn and comes ...",4.4,"Date, GMT","Luminous hands, Rotating Bezel, Luminous indices"
https://www.chrono24.com/longines/-----l16414-ss---2300207--id29662410.htm,l16414,Longines,conquest heritage,L1.641.4,2300207,Automatic,Automatic,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,[product code] 2300207 [Brand name] LONGINES [...,4.8,Chronograph,Unknown
https://www.chrono24.com/longines/conquest-l38354726--id31461374.htm,l38354726,Longines,conquest,L3.835.4.72.6,23326,Unknown,Unknown,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.6,Unknown,Unknown
https://www.chrono24.com/longines/longines-legend-diver-42mm-automatic-date-2017-full-set--id28997426.htm,l36744,Longines,legend diver,L3.674.4,Unknown,Automatic,Automatic,Steel,Steel,Leather,...,Buckle,Steel,Unknown,Unknown,Unknown,Unknown,Unknown,4.9,Date,Luminous hands
https://www.chrono24.com/longines/longines-conquest-2023-41mm-green-dial-steel-mens-watch-l38304026--id30497134.htm,l38304026,Longines,conquest,L3.830.4.02.6,Unknown,Automatic,Automatic,Steel,Steel,Steel,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,Item Longines Men's Watch Model # L3.830.4.02....,4.9,Unknown,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/longines-longines---l37284666-conquest-vhp-gmt-43-stainless-steel--carbon--bracelet-b--id31710938.htm,l37284666,Longines,Unknown,L37284666,Unknown,Unknown,Unknown,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,ロンジン - L3.728.4.66.6 コンクエスト V.H.P. GMT 43 ステンレ...,4.9,Unknown,Unknown
https://www.chrono24.com/longines/longines-longines-record-watch-l23214566--id25559997.htm,l23214566,Longines,record,L23214566,Unknown,Automatic,Automatic,Steel,Steel,Steel,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,brand Longines,4.9,Date,Unknown
https://www.chrono24.com/longines/longines-longines-dolcevita-black-dial-mens-watch-l57554576--id30283901.htm,l57554576,Longines,dolcevita,L57554576,Unknown,Unknown,Unknown,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown


In [ ]:
dbomelist_exp['Price']

url
https://www.chrono24.com/longines/hydroconquest-gmt--id31062563.htm                                                                                     €2,440 (= $2,702)
https://www.chrono24.com/longines/-----l16414-ss---2300207--id29662410.htm                                                                            ¥187,000 (= $1,320)
https://www.chrono24.com/longines/conquest-l38354726--id31461374.htm                                                                                    €2,991 (= $3,312)
https://www.chrono24.com/longines/longines-legend-diver-42mm-automatic-date-2017-full-set--id28997426.htm                                               €1,785 (= $1,977)
https://www.chrono24.com/longines/longines-conquest-2023-41mm-green-dial-steel-mens-watch-l38304026--id30497134.htm                                   $1,699 [Negotiable]
                                                                                                                                                  

In [ ]:
dbomelist_exp['PriceUSD']=[re.search(r'\$\d+(?:,(\d+))?',x).group() if re.search(r'\$\d+(?:,(\d+))?',x) else x for x in list(dbomelist_exp['Price'])]

In [ ]:
dbomelist_exp

,refnum,Brand,Model,Reference number,Dealer product code,Movement,Movement,Case material,Case material,Bracelet material,...,Clasp material,Bracelet length,Lug width,Buckle width,Bracelet thickness,data2,Seller rating,data3,data1,PriceUSD
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/longines/hydroconquest-gmt--id31062563.htm,l37904966,Longines,hydroconquest,L3.790.4.96.6,8694716006724,Automatic,Automatic,Steel,Steel,Steel,...,Unknown,Unknown,Unknown,Unknown,Unknown,"This model is original, new, unworn and comes ...",4.4,"Date, GMT","Luminous hands, Rotating Bezel, Luminous indices","$2,702"
https://www.chrono24.com/longines/-----l16414-ss---2300207--id29662410.htm,l16414,Longines,conquest heritage,L1.641.4,2300207,Automatic,Automatic,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,[product code] 2300207 [Brand name] LONGINES [...,4.8,Chronograph,Unknown,"$1,320"
https://www.chrono24.com/longines/conquest-l38354726--id31461374.htm,l38354726,Longines,conquest,L3.835.4.72.6,23326,Unknown,Unknown,Unknown,Unknown,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.6,Unknown,Unknown,"$3,312"
https://www.chrono24.com/longines/longines-legend-diver-42mm-automatic-date-2017-full-set--id28997426.htm,l36744,Longines,legend diver,L3.674.4,Unknown,Automatic,Automatic,Steel,Steel,Leather,...,Steel,Unknown,Unknown,Unknown,Unknown,Unknown,4.9,Date,Luminous hands,"$1,977"
https://www.chrono24.com/longines/longines-conquest-2023-41mm-green-dial-steel-mens-watch-l38304026--id30497134.htm,l38304026,Longines,conquest,L3.830.4.02.6,Unknown,Automatic,Automatic,Steel,Steel,Steel,...,Steel,Unknown,Unknown,Unknown,Unknown,Item Longines Men's Watch Model # L3.830.4.02....,4.9,Unknown,Unknown,"$1,699"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/longines-longines---l37284666-conquest-vhp-gmt-43-stainless-steel--carbon--bracelet-b--id31710938.htm,l37284666,Longines,Unknown,L37284666,Unknown,Unknown,Unknown,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,ロンジン - L3.728.4.66.6 コンクエスト V.H.P. GMT 43 ステンレ...,4.9,Unknown,Unknown,$898
https://www.chrono24.com/longines/longines-longines-record-watch-l23214566--id25559997.htm,l23214566,Longines,record,L23214566,Unknown,Automatic,Automatic,Steel,Steel,Steel,...,Steel,Unknown,Unknown,Unknown,Unknown,brand Longines,4.9,Date,Unknown,"$1,258"
https://www.chrono24.com/longines/longines-longines-dolcevita-black-dial-mens-watch-l57554576--id30283901.htm,l57554576,Longines,dolcevita,L57554576,Unknown,Unknown,Unknown,Steel,Steel,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,$768


In [ ]:
(pd.Series([1 if re.search(r'\$\d+(?:,(\d+))?',x) else x for x in list(dbomelist_exp['Price'])])==1).sum()

9505

In [ ]:
pd.Series([1 if re.search(r'\$\d+(?:,(\d+))?',x) else x for x in list(dbomelist_exp['Price'])]).unique()

array([1, 'Unknown', 'Price on request', 'Price on request [Negotiable]'],
      dtype=object)

In [ ]:
dbomelist_exp.loc[:,'PriceUSD']=dbomelist_exp['PriceUSD'].replace('Price on request [Negotiable]', 'Unknown')

In [ ]:
dbomelist_exp.loc[:,'PriceUSD']=dbomelist_exp['PriceUSD'].replace('Price on request', 'Unknown')

In [ ]:
#dbomelist_exp[100:150]

In [ ]:
dbomelist_exp['Seller rating'].unique()

array(['4.4', '4.8', '4.6', '4.9', ' ', '5.0', '4.7', '4.5', '4.3', '4.2',
       '4.1', '3.5', '3.6', '3.8', '3.9', '3.3', '3.7'], dtype=object)

In [ ]:
(dbomelist_exp.PriceUSD=='Unknown').sum()

200

In [ ]:
dbomelist_exp['NormalizedPrice']=0

In [ ]:
dbomelist_exp['Hierarchy_code']=''

In [ ]:
dbomelist_exp.columns

Index(['refnum', 'Brand', 'Model', 'Reference number', 'Dealer product code',
       'Movement', 'Movement', 'Case material', 'Case material',
       'Bracelet material', 'Bracelet material', 'Year of production',
       'Condition', 'Scope of delivery', 'Gender', 'Location', 'Price',
       'Availability', 'Movement', 'Movement', 'Caliber/movement',
       'Power reserve', 'Number of jewels', 'Frequency', 'Base caliber',
       'Case material', 'Case material', 'Case diameter', 'Water resistance',
       'Crystal', 'Dial', 'Thickness', 'Bezel material', 'Dial numerals',
       'Bracelet material', 'Bracelet material', 'Bracelet color', 'Clasp',
       'Clasp material', 'Bracelet length', 'Lug width', 'Buckle width',
       'Bracelet thickness', 'data2', 'Seller rating', 'data3', 'data1',
       'PriceUSD', 'NormalizedPrice', 'Hierarchy_code'],
      dtype='object')

In [ ]:
dbomelist_exp.T.duplicated()

refnum                 False
Brand                  False
Model                  False
Reference number       False
Dealer product code    False
Movement               False
Movement                True
Case material          False
Case material           True
Bracelet material      False
Bracelet material       True
Year of production     False
Condition              False
Scope of delivery      False
Gender                 False
Location               False
Price                  False
Availability           False
Movement                True
Movement                True
Caliber/movement       False
Power reserve          False
Number of jewels       False
Frequency              False
Base caliber           False
Case material           True
Case material           True
Case diameter          False
Water resistance       False
Crystal                False
Dial                   False
Thickness              False
Bezel material         False
Dial numerals          False
Bracelet mater

In [ ]:
dbomelist_exp=dbomelist_exp.T.drop_duplicates().T

In [ ]:
dbomelist_exp=dbomelist_exp.drop(['Dealer product code'],axis=1)

## Building of hierarchical groups

In [ ]:
def hierarchy_mapold(variation='default'):
  hilst=[]
  hilst1=list(dbomelist_exp['Brand'])
  hilst2=list(dbomelist_exp['Model'])
  hilst3=list(dbomelist_exp['refnum'])
  hilst4=list(dbomelist_exp['Year of production'])
  if variation!='default':
      hilst5=list(dbomelist_exp[variation])
  for ii in range(0,dbomelist_exp.shape[0]):
    if variation=='default':
      hilst.append(hilst1[ii]+'/'+hilst2[ii]+'/'+hilst3[ii]+'/'+'default')
    else:
      hilst.append(hilst1[ii]+'/'+hilst2[ii]+'/'+hilst3[ii]+'/'+hilst5[ii])
  dbomelist_exp['Hierarchy_code']=hilst

In [ ]:
def hierarchy_map(variation='default'):
  hilst=[]
  hilst1=list(dbomelist_exp['Brand'])
  hilst2=list(dbomelist_exp['Model'])
  hilst3=list(dbomelist_exp['refnum'])
  hilst4=list(dbomelist_exp['Year of production'])
  if variation!='default':
      hilst5=list(dbomelist_exp[variation])
  for ii in range(0,dbomelist_exp.shape[0]):
    if variation=='default':
      try:
        hilst.append(dict_hiercode_models[hilst3[ii]]+'/'+'default')
      except Exception as err:
        hilst.append(hilst1[ii]+'/'+hilst2[ii]+'/'+hilst3[ii]+'/'+'default')
    else:
      try:
        hilst.append(dict_hiercode_models[hilst3[ii]]+hilst5[ii])
      except Exception as err:
        hilst.append(hilst1[ii]+'/'+hilst2[ii]+'/'+hilst3[ii]+'/'+hilst5[ii])

  dbomelist_exp['Hierarchy_code']=hilst

In [ ]:
hierarchy_map()

In [ ]:
dbomelist_exp.Model.unique()

array(['hydroconquest', 'conquest heritage', 'conquest', 'legend diver',
       'master collection', 'Unknown', 'primaluna', 'heritage', 'spirit',
       'record', 'elegant', 'la grande classique', 'oposition', 'lyre',
       'dolcevita', 'lindbergh hour angle', 'conquest classic',
       'avigation', 'saint imier', 'flagship', 'présence', 'equestrian',
       'symphonette', 'evidenza', 'admiral', 'column wheel chronograph',
       'twenty four hours', 'flagship heritage', 'grande vitesse',
       'présence heritage'], dtype=object)

In [ ]:
lstconv=['.'.join(x.split(',')) if x!='Unknown' else x for x in dbomelist_exp.PriceUSD]
dbomelist_exp.PriceUSD=lstconv

In [ ]:
lstconv=[x[1:] for x in dbomelist_exp.PriceUSD]
dbomelist_exp.PriceUSD=lstconv

In [ ]:
dbomelist_exp.loc[:,'PriceUSD']=dbomelist_exp.PriceUSD.replace('nknown','Unknown')

In [ ]:
dbomelist_exp.rename(columns={'PriceUSD':'PriceUSDthou'},inplace=True)

In [ ]:
dbeach_modlst=[]
scaler=MinMaxScaler()
cnt=dbomelist_exp.Model.unique()
for mm in cnt:
    print(mm)
    dbeach_mod=dbomelist_exp[dbomelist_exp.Model==mm]
    if len(np.array(dbeach_mod[dbeach_mod['PriceUSDthou']!='Unknown'].PriceUSDthou))!=0:
       meanprice=np.mean(np.array(dbeach_mod[dbeach_mod['PriceUSDthou']!='Unknown'].PriceUSDthou.astype(np.float32)))
       dbeach_mod.loc[:,'NormalizedPrice']=dbeach_mod['PriceUSDthou'].replace('Unknown',str(meanprice)).astype(np.float32)
       dbeach_mod.loc[:,'NormalizedPrice']=scaler.fit_transform(dbeach_mod[['NormalizedPrice']])
    else:
       dbeach_mod.loc[:,'NormalizedPrice']=0.5
 #   dbeach_mod.loc[:,'NormalizedPrice']=dbeach_mod['NormalizedPrice'].astype(np.float32)
    dbeach_modlst.append(dbeach_mod)
dbomelist_exp1=pd.concat(dbeach_modlst,axis=0)

hydroconquest
conquest heritage
conquest
legend diver
master collection
Unknown
primaluna
heritage
spirit
record
elegant
la grande classique
oposition
lyre
dolcevita
lindbergh hour angle
conquest classic
avigation
saint imier
flagship
présence
equestrian
symphonette
evidenza
admiral
column wheel chronograph
twenty four hours
flagship heritage
grande vitesse
présence heritage


In [ ]:
dbomelist_expnorpr=dbomelist_exp1.copy()
dbomelist_expnorpr

,refnum,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,Scope of delivery,...,Lug width,Buckle width,Bracelet thickness,data2,Seller rating,data3,data1,PriceUSDthou,NormalizedPrice,Hierarchy_code
url,,,,,,,,,,,,,,,,,,,,,
https://www.chrono24.com/longines/hydroconquest-gmt--id31062563.htm,l37904966,Longines,hydroconquest,L3.790.4.96.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)","Original box, original papers",...,Unknown,Unknown,Unknown,"This model is original, new, unworn and comes ...",4.4,"Date, GMT","Luminous hands, Rotating Bezel, Luminous indices",2.702,0.001705,Longines/hydroconquest gmt/l37904966/default
https://www.chrono24.com/longines/longines-hydroconquest-bp-2023--id31772173.htm,l37904066,Longines,hydroconquest,L3.790.4.06.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)","Original box, original papers",...,Unknown,Unknown,Unknown,hydroconquest gmt The hydroconquest collection...,4.9,Unknown,,2.990,0.001994,Longines/l3 790 4 06 6/l37904066/default
https://www.chrono24.com/longines/longines-hydroconquest--id31542335.htm,l36424566,Longines,hydroconquest,L3.642.4.56.6,Automatic,Steel,Steel,2013,Very good (Worn with little to no signs of wear),"Original box, original papers",...,Unknown,Unknown,Unknown,The Longines HydroConquest L3.642.4.56.6 has a...,4.9,Date,"Central seconds, Luminous numerals, Luminous h...",1.107,0.000107,Longines/hydroconquest/l36424566/default
https://www.chrono24.com/longines/hydroconquest--id30844794.htm,l36424566,Longines,hydroconquest,L3.7814,Automatic,Steel,Steel,2023,Very good (Worn with little to no signs of wear),"Original box, original papers",...,Unknown,Unknown,Unknown,I am selling beautiful Longines hidroconquest ...,,Date,Unknown,1.661,0.000662,Longines/hydroconquest/l36424566/default
https://www.chrono24.com/longines/longines-hydroconquest-gmt-olive--id31377331.htm,l37904062,Longines,hydroconquest,L3.790.4.06.2,Automatic,Steel,Textile,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",...,Unknown,Unknown,Unknown,The New Longines Hydroconquest GMT's are leavi...,5.0,"Date, GMT","Luminous numerals, Luminous hands, Rotating Be...",2.270,0.001273,Longines/hydroconquest gmt/l37904062/default
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
https://www.chrono24.com/longines/longines-heritage-l47858732--id19346034.htm,l47858732,Longines,présence heritage,L4.785.8.73.2,Automatic,Rose gold,Unknown,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",...,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,Unknown,0.280976,Longines/heritage/l47858732/default
https://www.chrono24.com/longines/longines-heritage--id31858432.htm,l47858732,Longines,présence heritage,L4.785.8.73.2,Automatic,Rose gold,Unknown,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",...,Unknown,Unknown,Unknown,Welcome to our shop! All our watches sold on C...,4.7,Unknown,Unknown,Unknown,0.280976,Longines/heritage/l47858732/default
https://www.chrono24.com/longines/longines-heritage--id31858434.htm,l47858732,Longines,présence heritage,L4.785.8.73.2,Automatic,Rose gold,Unknown,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",...,Unknown,Unknown,Unknown,Welcome to our shop! All our watches sold on C...,4.8,Unknown,Unknown,Unknown,0.280976,Longines/heritage/l47858732/default


In [ ]:
##########

In [ ]:
dbmod0.reset_index().to_excel("models.xlsx")

In [ ]:
hierarchy_map()
dbeach_modlst=[]
scaler=MinMaxScaler()
cnt=dbomelist_exp.Model.unique()
for mm in cnt:
    print(mm)
    dbeach_mod=dbomelist_exp[dbomelist_exp.Model==mm]
    if len(np.array(dbeach_mod[dbeach_mod['PriceUSDthou']!='Unknown'].PriceUSDthou))!=0:
       meanprice=np.mean(np.array(dbeach_mod[dbeach_mod['PriceUSDthou']!='Unknown'].PriceUSDthou.astype(np.float32)))
       dbeach_mod.loc[:,'NormalizedPrice']=dbeach_mod['PriceUSDthou'].replace('Unknown',str(meanprice)).astype(np.float32)
       dbeach_mod.loc[:,'NormalizedPrice']=scaler.fit_transform(dbeach_mod[['NormalizedPrice']])
       dbeach_mod.loc[:,'PriceUSDthou']=dbeach_mod['PriceUSDthou'].replace('Unknown',str(meanprice))
    else:
       dbeach_mod.loc[:,'NormalizedPrice']=0.5
#    dbeach_mod.loc[:,'NormalizedPrice']=dbeach_mod['NormalizedPrice'].astype(np.float32)
    dbeach_modlst.append(dbeach_mod)
dbomelist_exp1=pd.concat(dbeach_modlst,axis=0)
dbomelist_expnorpr0=dbomelist_exp1.copy()
#dbomelist_expnorpr0.reset_index().to_excel("listings def hierarchy.xlsx")

hydroconquest
conquest heritage
conquest
legend diver
master collection
Unknown
primaluna
heritage
spirit
record
elegant
la grande classique
oposition
lyre
dolcevita
lindbergh hour angle
conquest classic
avigation
saint imier
flagship
présence
equestrian
symphonette
evidenza
admiral
column wheel chronograph
twenty four hours
flagship heritage
grande vitesse
présence heritage


In [ ]:
hierarchy_map(VARIANT)
dbeach_modlst=[]
scaler=MinMaxScaler()
cnt=dbomelist_exp.Model.unique()
for mm in cnt:
    print(mm)
    dbeach_mod=dbomelist_exp[dbomelist_exp.Model==mm]
    if len(np.array(dbeach_mod[dbeach_mod['PriceUSDthou']!='Unknown'].PriceUSDthou))!=0:
       meanprice=np.mean(np.array(dbeach_mod[dbeach_mod['PriceUSDthou']!='Unknown'].PriceUSDthou.astype(np.float32)))
       dbeach_mod.loc[:,'NormalizedPrice']=dbeach_mod['PriceUSDthou'].replace('Unknown',str(meanprice)).astype(np.float32)
       dbeach_mod.loc[:,'NormalizedPrice']=scaler.fit_transform(dbeach_mod[['NormalizedPrice']])
       dbeach_mod.loc[:,'PriceUSDthou']=dbeach_mod['PriceUSDthou'].replace('Unknown',str(meanprice))
    else:
       dbeach_mod.loc[:,'NormalizedPrice']=0.5
#    dbeach_mod.loc[:,'NormalizedPrice']=dbeach_mod['NormalizedPrice'].astype(np.float32)
    dbeach_modlst.append(dbeach_mod)
dbomelist_exp1=pd.concat(dbeach_modlst,axis=0)
dbomelist_expnorpr1=dbomelist_exp1.copy()
#dbomelist_expnorpr1.reset_index().to_excel("listings with variant hierarchy.xlsx")

hydroconquest
conquest heritage
conquest
legend diver
master collection
Unknown
primaluna
heritage
spirit
record
elegant
la grande classique
oposition
lyre
dolcevita
lindbergh hour angle
conquest classic
avigation
saint imier
flagship
présence
equestrian
symphonette
evidenza
admiral
column wheel chronograph
twenty four hours
flagship heritage
grande vitesse
présence heritage


In [ ]:
#########

In [ ]:
dbmod0jsn=[]
dbmod0r=dbmod0.reset_index()
for jj in list(dbmod0r.index):
   dbmod0jsn.append(dbmod0r.loc[jj,:].to_dict())
with open("mongopre_modelsv1.json", "w", encoding="utf-8") as file:
    file.write(json.dumps(dbmod0jsn, indent=4))

In [ ]:
dbomelist_expnorpr0jsn=[]
dbomelist_expnorpr0r=dbomelist_expnorpr0.reset_index()
for jj in list(dbomelist_expnorpr0r.index):
   dbomelist_expnorpr0jsn.append(dbomelist_expnorpr0r.loc[jj,:].to_dict())
with open(f"mongopre_listingsdef{BRND.lower()}v1.json", "w", encoding="utf-8") as file:
    file.write(json.dumps(dbomelist_expnorpr0jsn, indent=4))

In [ ]:
dbomelist_expnorpr1jsn=[]
dbomelist_expnorpr1r=dbomelist_expnorpr1.reset_index()
for jj in list(dbomelist_expnorpr1r.index):
   dbomelist_expnorpr1jsn.append(dbomelist_expnorpr1r.loc[jj,:].to_dict())
#with open("f"mongopre_listingsonvar{BRND.lower()}{VARIANT.lower()}v1.json", "w", encoding="utf-8") as file:
#    file.write(json.dumps(dbomelist_expnorpr1jsn, indent=4))

## Omega and other brands catalog building

### Building of a representative of a hierarchical group with most probable features

In [ ]:
dbomelist_expnorpr0=dbomelist_expnorpr0.reset_index().set_index('Hierarchy_code').sort_index()

In [ ]:
dbomelist_feamap=dbomelist_expnorpr0.drop(['refnum', 'Brand', 'Model', 'Reference number', 'data2','PriceUSDthou','Price','url'],axis=1)

In [ ]:
dbomelist_feamap

,Movement,Case material,Bracelet material,Year of production,Condition,Scope of delivery,Gender,Location,Availability,Caliber/movement,...,Clasp,Clasp material,Bracelet length,Lug width,Buckle width,Bracelet thickness,Seller rating,data3,data1,NormalizedPrice
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,Manual winding,Yellow gold,Leather,1950,Good (Light signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"United States of America, New York, New York",Item is in stock,Unknown,...,Buckle,Gold/Steel,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.000276
Longines/ admiral automatic/1200/default,Automatic,Yellow gold,Unknown,1967,Good (Light signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"Italy, ROMA",Item is in stock,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown,0.000973
Longines/ admiral automatic/1200/default,Automatic,Yellow gold,Unknown,Unknown,Fair (Obvious signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"United States of America, New Jersey, West Orange",Item is in stock,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown,0.639076
Longines/ automatic/l49224126/default,Automatic,Steel,Steel,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Unknown,"Hong Kong, HK",Item needs to be procured,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.4,Unknown,Unknown,0.000424
Longines/ automatic/l49224126/default,Automatic,Steel,Steel,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",Unknown,"Germany, Düsseldorf",Item available on request,Unknown,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.7,Date,Unknown,0.000545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/zulu time automatic 39 mm spirit/l38024532/default,Automatic,Steel,Leather,2023,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"United Kingdom, London",Item needs to be procured,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.3,Unknown,Unknown,0.083027
Longines/zulu time automatic 39 mm spirit/l38024532/default,Automatic,Steel,Leather,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Hong Kong, Hong Kong",Item needs to be procured,Unknown,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.06798
Longines/zulu time automatic 39 mm spirit/l38024532/default,Automatic,Steel,Leather,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Germany, Köln",Item needs to be procured,L888,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.065025


In [ ]:
#dbomelist_feamap.info()

In [ ]:
dbomelist_feamap.fillna(0.5,inplace=True)

In [ ]:
dbomelist_feamap[dbomelist_feamap['NormalizedPrice'].isna()]

,Movement,Case material,Bracelet material,Year of production,Condition,Scope of delivery,Gender,Location,Availability,Caliber/movement,...,Clasp,Clasp material,Bracelet length,Lug width,Buckle width,Bracelet thickness,Seller rating,data3,data1,NormalizedPrice
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,


In [ ]:
kproto = KPrototypes(n_clusters=1, init='Cao',random_state=42)
clusters=kproto.fit_predict(dbomelist_feamap,categorical=list(range(0,(dbomelist_feamap.shape[1]-1))))

In [ ]:
kproto.cluster_centroids_

array([['0.0958013398802166', 'Automatic', 'Steel', 'Steel', '2023',
        'New (Brand new, without any signs of wear)',
        'Original box, original papers', "Men's watch/Unisex",
        'United Kingdom, London', 'Item needs to be procured', 'Unknown',
        'Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown',
        'Sapphire crystal', 'Black', 'Unknown', 'Unknown', 'Unknown',
        'Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown',
        'Unknown', '4.9', 'Unknown', 'Unknown']], dtype='<U42')

In [ ]:
lis=list(kproto.cluster_centroids_[0,1:])
lis.append(kproto.cluster_centroids_[0,0])
centr=np.array(lis).reshape(1,-1)
centrdb=pd.DataFrame(centr,columns=list(dbomelist_feamap.columns))
dbomelist_feamap=pd.concat([dbomelist_feamap,centrdb],axis=0)
dbomelist_feamap

,Movement,Case material,Bracelet material,Year of production,Condition,Scope of delivery,Gender,Location,Availability,Caliber/movement,...,Clasp,Clasp material,Bracelet length,Lug width,Buckle width,Bracelet thickness,Seller rating,data3,data1,NormalizedPrice
Longines/ admiral automatic/1200/default,Manual winding,Yellow gold,Leather,1950,Good (Light signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"United States of America, New York, New York",Item is in stock,Unknown,...,Buckle,Gold/Steel,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.000276
Longines/ admiral automatic/1200/default,Automatic,Yellow gold,Unknown,1967,Good (Light signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"Italy, ROMA",Item is in stock,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown,0.000973
Longines/ admiral automatic/1200/default,Automatic,Yellow gold,Unknown,Unknown,Fair (Obvious signs of wear or scratches),"No original box, no original papers",Men's watch/Unisex,"United States of America, New Jersey, West Orange",Item is in stock,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown,0.639076
Longines/ automatic/l49224126/default,Automatic,Steel,Steel,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Unknown,"Hong Kong, HK",Item needs to be procured,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.4,Unknown,Unknown,0.000424
Longines/ automatic/l49224126/default,Automatic,Steel,Steel,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",Unknown,"Germany, Düsseldorf",Item available on request,Unknown,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.7,Date,Unknown,0.000545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/zulu time automatic 39 mm spirit/l38024532/default,Automatic,Steel,Leather,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Hong Kong, Hong Kong",Item needs to be procured,Unknown,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.06798
Longines/zulu time automatic 39 mm spirit/l38024532/default,Automatic,Steel,Leather,Unknown,"New (Brand new, without any signs of wear)","Original box, original papers",Men's watch/Unisex,"Germany, Köln",Item needs to be procured,L888,...,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.065025
Longines/zulu time automatic 39 mm spirit/l38024532/default,Automatic,Steel,Leather,2023,"Unworn (Mint condition, without signs of wear)","Original box, original papers",Men's watch/Unisex,"Poland, Warszawa",Item needs to be procured,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown,0.076847
Longines/アドミラル 5スター デイデイト 自動巻き/l76342/default,Automatic,Steel,Steel,Unknown,"Unworn (Mint condition, without signs of wear)","No original box, no original papers",Men's watch/Unisex,"Turkey, istanbul",Item is in stock,Unknown,...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.5,Unknown,Unknown,0.755503


In [ ]:
gowsimm=gower.gower_matrix(dbomelist_feamap)

In [ ]:
len(gowsimm[-1,:-1])

9705

In [ ]:
dbomelist_expnorpr0['Dissimilarity']=gowsimm[-1,:-1]

In [ ]:
dbomelist_expnorpr0.reset_index(inplace=True)
dbomelist_expnorpr0['sub_idx']=dbomelist_expnorpr0.groupby("Hierarchy_code").cumcount()
dbomelist_expnorpr0=dbomelist_expnorpr0.set_index(['Hierarchy_code','sub_idx'])

In [ ]:
##########

In [ ]:
idxlev0=dbomelist_expnorpr0.index.get_level_values(0).to_series().unique()

In [ ]:
dbomelist_expnorpr0.columns

Index(['url', 'refnum', 'Brand', 'Model', 'Reference number', 'Movement',
       'Case material', 'Bracelet material', 'Year of production', 'Condition',
       'Scope of delivery', 'Gender', 'Location', 'Price', 'Availability',
       'Caliber/movement', 'Power reserve', 'Number of jewels', 'Frequency',
       'Base caliber', 'Case diameter', 'Water resistance', 'Crystal', 'Dial',
       'Thickness', 'Bezel material', 'Dial numerals', 'Bracelet color',
       'Clasp', 'Clasp material', 'Bracelet length', 'Lug width',
       'Buckle width', 'Bracelet thickness', 'data2', 'Seller rating', 'data3',
       'data1', 'PriceUSDthou', 'NormalizedPrice', 'Dissimilarity'],
      dtype='object')

In [ ]:
lagg=[]
for ee in idxlev0:
    listagg=dbomelist_expnorpr0.loc[(ee,),:]
    listaggone=pd.DataFrame(listagg.loc[listagg['Dissimilarity'].idxmin()]).T
    listaggone['Hierarchy_code']=ee
    lagg.append(listaggone.set_index('Hierarchy_code'))

In [ ]:
columnames=list(dbomelist_expnorpr0.columns[5:-3])

In [ ]:
columnames.remove('Price')

In [ ]:
columnames.remove('data2')

In [ ]:
columnames

['Movement',
 'Case material',
 'Bracelet material',
 'Year of production',
 'Condition',
 'Scope of delivery',
 'Gender',
 'Location',
 'Availability',
 'Caliber/movement',
 'Power reserve',
 'Number of jewels',
 'Frequency',
 'Base caliber',
 'Case diameter',
 'Water resistance',
 'Crystal',
 'Dial',
 'Thickness',
 'Bezel material',
 'Dial numerals',
 'Bracelet color',
 'Clasp',
 'Clasp material',
 'Bracelet length',
 'Lug width',
 'Buckle width',
 'Bracelet thickness',
 'Seller rating',
 'data3',
 'data1']

In [ ]:
dagg={}
for xx in columnames:
   lagg1=[]
   for  ee in idxlev0:
    listagg1=dbomelist_expnorpr0.loc[(ee,),:]
    lagg1.append(pd.DataFrame(listagg1.groupby(xx)[xx].count()).to_dict())
   dagg[xx] =lagg1

In [ ]:
dagg['Year of production']

[{'Year of production': {'1950': 1, '1967': 1, 'Unknown': 1}},
 {'Year of production': {'2023': 4, 'Unknown': 2}},
 {'Year of production': {'2023': 8, 'Unknown': 3}},
 {'Year of production': {'2018': 1, '2019': 1}},
 {'Year of production': {'2023': 5, 'Unknown': 2}},
 {'Year of production': {'2023': 4, 'Unknown': 2}},
 {'Year of production': {'2023': 7, 'Unknown': 4}},
 {'Year of production': {'2023': 1, 'Unknown': 2}},
 {'Year of production': {'2022': 1, '2023': 5}},
 {'Year of production': {'2023': 3}},
 {'Year of production': {'Unknown': 5}},
 {'Year of production': {'2023': 1, 'Unknown': 3}},
 {'Year of production': {'2011': 1, 'Unknown': 2}},
 {'Year of production': {'2017': 1, '2023': 3, 'Unknown': 1}},
 {'Year of production': {'1950': 1, '1959': 1, '1960': 1, 'Unknown': 1}},
 {'Year of production': {'2023': 3}},
 {'Year of production': {'2023': 4}},
 {'Year of production': {'1966': 1, '1968': 3, 'Unknown': 1}},
 {'Year of production': {'1950': 2, '1960': 1}},
 {'Year of producti

In [ ]:
daggmax={}
for xx in columnames:
   lagg2=[]
   for  ee in idxlev0:
    listagg2=dbomelist_expnorpr0.loc[(ee,),:]
    if listagg2.groupby(xx)[xx].count().drop(['Unknown'],errors='ignore').size!=0:
     lagg2.append((listagg2.groupby(xx)[xx].count().drop(['Unknown'],errors='ignore')).idxmax())
    else:
     lagg2.append('Unknown')
   daggmax[xx] =lagg2

In [ ]:
(pd.Series(daggmax['data1'])=='Unknown').sum()

780

In [ ]:
(pd.Series(daggmax['Movement'])=='Unknown').sum()

19

In [ ]:
ome_catalog=pd.concat(lagg,axis=0)
#ome_catalog

In [ ]:
for key0,value0 in dagg.items():
    ome_catalog[key0+'0']=value0

for key1,value1 in daggmax.items():
    ome_catalog[key1+'1']=value1

ome_catalog

/tmp/ipykernel_43/2332726937.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ome_catalog[key1+'1']=value1
/tmp/ipykernel_43/2332726937.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ome_catalog[key1+'1']=value1
/tmp/ipykernel_43/2332726937.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  

,url,refnum,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color1,Clasp1,Clasp material1,Bracelet length1,Lug width1,Buckle width1,Bracelet thickness1,Seller rating1,data31,data11
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,https://www.chrono24.com/longines/admiral--id2...,1200,Longines,admiral,1200,Automatic,Yellow gold,Unknown,1967,Good (Light signs of wear or scratches),...,Black,Buckle,Gold/Steel,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown
Longines/ automatic/l49224126/default,https://www.chrono24.com/longines/longines-aut...,l49224126,Longines,Unknown,L4.922.4.12.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,Steel,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.8,Date,"Display back, Central seconds"
Longines/ conquest automatic/l34304076/default,https://www.chrono24.com/longines/longines-con...,l34304076,Longines,conquest,L3.430.4.07.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,Steel,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.9,Date,"Display back, Central seconds, Luminous hands,..."
Longines/ conquest コンクエスト/l33794/default,https://www.chrono24.com/longines/longines-con...,l33794,Longines,conquest,L3.379.4,Quartz,Steel,Steel,2018,Fair (Obvious signs of wear or scratches),...,Steel,"Fold clasp, hidden",Steel,136 mm (68 mm / 68 mm),17 mm Size guide,17 mm,4 mm,4.5,"Chronograph, Date","Screw-Down Crown, Luminous indices"
Longines/ conquest/l33763877/default,https://www.chrono24.com/longines/longines-con...,l33763877,Longines,conquest,L3.376.3.87.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,Gold/Steel,Double-fold clasp,Steel,Unknown,15 mm Size guide,Unknown,Unknown,4.3,Date,Gemstones and/or diamonds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/with box and dolcevita/l52550715/default,https://www.chrono24.com/longines/dolcevita-l5...,l52550715,Longines,dolcevita,L5.255.0.71.5,Unknown,Unknown,Unknown,2023,"New (Brand new, without any signs of wear)",...,Red,Buckle,Unknown,Unknown,Unknown,Unknown,Unknown,4.8,Unknown,Small seconds
Longines/with box and dolcevita/l55129870/default,https://www.chrono24.com/longines/longines-wit...,l55129870,Longines,dolcevita,L5.512.9.87.0,Automatic,Rose gold,Unknown,2023,"Unworn (Mint condition, without signs of wear)",...,Black,Buckle,Unknown,Unknown,Unknown,Unknown,Unknown,4.6,Unknown,Unknown
Longines/with box and elegant collection/l47878124/default,https://www.chrono24.com/longines/elegant-coll...,l47878124,Longines,elegant,L4.787.8.12.4,Automatic,Rose gold,Leather,2023,"New (Brand new, without any signs of wear)",...,Black,Buckle,Unknown,Unknown,Unknown,Unknown,Unknown,4.3,Date,"Display back, Central seconds"


In [ ]:
(ome_catalog.data1=='Unknown').sum()

1372

In [ ]:
ome_catalog.refnum.nunique()

1404

In [ ]:
ome_catalog[ome_catalog['PriceUSDthou']=='Unknown']

,url,refnum,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color1,Clasp1,Clasp material1,Bracelet length1,Lug width1,Buckle width1,Bracelet thickness1,Seller rating1,data31,data11
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,


In [ ]:
ome_catalog.drop(['Price'],axis=1,inplace=True)

In [ ]:
ome_catalog2=ome_catalog.reset_index().set_index('refnum')
ome_catalog2=ome_catalog2[ome_catalog2.reset_index().groupby('refnum')['Model'].count()>1]
ome_catalog2=ome_catalog2.reset_index()
ome_catalog2=ome_catalog2.set_index('Hierarchy_code')

idxcat=list(ome_catalog2.index)
for yy in idxcat:
    if (ome_catalog2.loc[yy,'refnum']).lower()!=(ome_catalog2.loc[yy,'Reference number']).lower():
          ome_catalog2.drop([yy],axis=0,inplace=True)

/tmp/ipykernel_43/2675978694.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ome_catalog2=ome_catalog2[ome_catalog2.reset_index().groupby('refnum')['Model'].count()>1]


In [ ]:
ome_catalog1=ome_catalog.reset_index().set_index('refnum')
ome_catalog1=ome_catalog1[ome_catalog1.reset_index().groupby('refnum')['Model'].count()==1]
ome_catalog1=ome_catalog1.reset_index()
ome_catalog1=ome_catalog1.set_index('Hierarchy_code')

ome_catalog=pd.concat([ome_catalog1,ome_catalog2],axis=0)

/tmp/ipykernel_43/3434034556.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ome_catalog1=ome_catalog1[ome_catalog1.reset_index().groupby('refnum')['Model'].count()==1]


In [ ]:
ome_catalog

,refnum,url,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color1,Clasp1,Clasp material1,Bracelet length1,Lug width1,Buckle width1,Bracelet thickness1,Seller rating1,data31,data11
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,1200,https://www.chrono24.com/longines/admiral--id2...,Longines,admiral,1200,Automatic,Yellow gold,Unknown,1967,Good (Light signs of wear or scratches),...,Black,Buckle,Gold/Steel,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown
Longines/ automatic/l49224126/default,l49224126,https://www.chrono24.com/longines/longines-aut...,Longines,Unknown,L4.922.4.12.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,Steel,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.8,Date,"Display back, Central seconds"
Longines/ conquest automatic/l34304076/default,l34304076,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.430.4.07.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,Steel,Fold clasp,Steel,Unknown,Unknown,Unknown,Unknown,4.9,Date,"Display back, Central seconds, Luminous hands,..."
Longines/ conquest コンクエスト/l33794/default,l33794,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.379.4,Quartz,Steel,Steel,2018,Fair (Obvious signs of wear or scratches),...,Steel,"Fold clasp, hidden",Steel,136 mm (68 mm / 68 mm),17 mm Size guide,17 mm,4 mm,4.5,"Chronograph, Date","Screw-Down Crown, Luminous indices"
Longines/ conquest/l33763877/default,l33763877,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.376.3.87.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,Gold/Steel,Double-fold clasp,Steel,Unknown,15 mm Size guide,Unknown,Unknown,4.3,Date,Gemstones and/or diamonds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/Unknown/8888/default,8888,https://www.chrono24.com/longines/longines-rar...,Longines,Unknown,8888,Manual winding,Steel,Leather,Unknown,Good (Light signs of wear or scratches),...,Black,Buckle,Steel,190 mm (115 mm / 75 mm),18 mm Size guide,18 mm,2 mm,4.9,Unknown,Small seconds
Longines/Unknown/9002/default,9002,https://www.chrono24.com/longines/longines----...,Longines,Unknown,9002,Automatic,Unknown,Steel,Unknown,Fair (Obvious signs of wear or scratches),...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.8,Unknown,Unknown
Longines/conquest/9002/default,9002,https://www.chrono24.com/longines/longines-con...,Longines,conquest,9002,Automatic,Gold/Steel,Unknown,Unknown,Good (Light signs of wear or scratches),...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,5.0,Unknown,Unknown


In [ ]:
idxcat=list(ome_catalog.index)
for xx in columnames:
    for yy in idxcat:
      if ome_catalog.loc[yy,xx]=='Unknown':
        xx1=xx+'1'
        cc=ome_catalog.loc[yy,xx1]
        ome_catalog.loc[[yy],[xx]]=cc

In [ ]:
ome_catalog[ome_catalog['data3']=='Unknown']

,refnum,url,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color1,Clasp1,Clasp material1,Bracelet length1,Lug width1,Buckle width1,Bracelet thickness1,Seller rating1,data31,data11
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,1200,https://www.chrono24.com/longines/admiral--id2...,Longines,admiral,1200,Automatic,Yellow gold,Leather,1967,Good (Light signs of wear or scratches),...,Black,Buckle,Gold/Steel,Unknown,Unknown,Unknown,Unknown,,Unknown,Unknown
Longines/ l4 922 1 12 7/l49221127/default,l49221127,https://www.chrono24.com/longines/longines-l49...,Longines,Unknown,L49221127,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.3,Unknown,Unknown
Longines/ presence/l43221117/default,l43221117,https://www.chrono24.com/longines/longines-pre...,Longines,présence,L4.322.1.11.7,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.3,Unknown,Unknown
Longines/ クォーツ/l37014766/default,l37014766,https://www.chrono24.com/longines/longines-lon...,Longines,Unknown,L3.701.4.76.6,Quartz,Steel,Steel,Unknown,"New (Brand new, without any signs of wear)",...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.9,Unknown,Unknown
Longines/ プリマルナ pgコンビ d ssxpg クォーツ/l81095796/default,l81095796,https://www.chrono24.com/longines/longines-lon...,Longines,primaluna,L81095796,Quartz,Rose gold,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,Gold/Steel,Double-fold clasp,Steel,Unknown,14 mm Size guide,Unknown,Unknown,4.9,Unknown,Gemstones and/or diamonds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/Unknown/8888/default,8888,https://www.chrono24.com/longines/longines-rar...,Longines,Unknown,8888,Manual winding,Steel,Leather,1966,Good (Light signs of wear or scratches),...,Black,Buckle,Steel,190 mm (115 mm / 75 mm),18 mm Size guide,18 mm,2 mm,4.9,Unknown,Small seconds
Longines/Unknown/9002/default,9002,https://www.chrono24.com/longines/longines----...,Longines,Unknown,9002,Automatic,Unknown,Steel,Unknown,Fair (Obvious signs of wear or scratches),...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,4.8,Unknown,Unknown
Longines/conquest/9002/default,9002,https://www.chrono24.com/longines/longines-con...,Longines,conquest,9002,Automatic,Gold/Steel,Unknown,1950,Good (Light signs of wear or scratches),...,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,5.0,Unknown,Unknown


In [ ]:
idxcat=list(ome_catalog.index)
for xx in columnames:
  try:
    cc=(ome_catalog.groupby(xx)[xx].count().drop(['Unknown'],errors='ignore')).idxmax()
    for yy in idxcat:
      if ome_catalog.loc[yy,xx]=='Unknown':
        ome_catalog.loc[[yy],[xx]]=cc
  except:
    _=0
    print(f'in column {xx}-all Unknowns')

In [ ]:
for xx in columnames:
   ome_catalog.drop([xx+'1'],axis=1,inplace=True)

In [ ]:
ome_catalog

,refnum,url,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color0,Clasp0,Clasp material0,Bracelet length0,Lug width0,Buckle width0,Bracelet thickness0,Seller rating0,data30,data10
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,1200,https://www.chrono24.com/longines/admiral--id2...,Longines,admiral,1200,Automatic,Yellow gold,Leather,1967,Good (Light signs of wear or scratches),...,"{'Bracelet color': {'Black': 1, 'Unknown': 2}}","{'Clasp': {'Buckle': 1, 'Unknown': 2}}","{'Clasp material': {'Gold/Steel': 1, 'Unknown'...",{'Bracelet length': {'Unknown': 3}},{'Lug width': {'Unknown': 3}},{'Buckle width': {'Unknown': 3}},{'Bracelet thickness': {'Unknown': 3}},"{'Seller rating': {' ': 2, '4.9': 1}}",{'data3': {'Unknown': 3}},{'data1': {'Unknown': 3}}
Longines/ automatic/l49224126/default,l49224126,https://www.chrono24.com/longines/longines-aut...,Longines,Unknown,L4.922.4.12.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Silver': 1, 'Steel': 5}}","{'Clasp': {'Fold clasp': 5, 'Unknown': 1}}","{'Clasp material': {'Steel': 4, 'Unknown': 2}}",{'Bracelet length': {'Unknown': 6}},{'Lug width': {'Unknown': 6}},{'Buckle width': {'Unknown': 6}},{'Bracelet thickness': {'Unknown': 6}},"{'Seller rating': {'4.3': 1, '4.4': 1, '4.6': ...","{'data3': {'Date': 4, 'Unknown': 2}}","{'data1': {'Display back, Central seconds': 1,..."
Longines/ conquest automatic/l34304076/default,l34304076,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.430.4.07.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Steel': 10, 'Unknown': 1}}","{'Clasp': {'Fold clasp': 9, 'Unknown': 2}}","{'Clasp material': {'Steel': 8, 'Unknown': 3}}",{'Bracelet length': {'Unknown': 11}},{'Lug width': {'Unknown': 11}},{'Buckle width': {'Unknown': 11}},{'Bracelet thickness': {'Unknown': 11}},"{'Seller rating': {'4.3': 1, '4.4': 1, '4.5': ...","{'data3': {'Date': 2, 'Unknown': 9}}","{'data1': {'Display back, Central seconds, Lum..."
Longines/ conquest コンクエスト/l33794/default,l33794,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.379.4,Quartz,Steel,Steel,2018,Fair (Obvious signs of wear or scratches),...,"{'Bracelet color': {'Steel': 1, 'Unknown': 1}}","{'Clasp': {'Fold clasp, hidden': 2}}",{'Clasp material': {'Steel': 2}},{'Bracelet length': {'136 mm (68 mm / 68 mm)':...,"{'Lug width': {'17 mm Size guide': 1, 'Unknown...","{'Buckle width': {'17 mm': 1, 'Unknown': 1}}","{'Bracelet thickness': {'4 mm': 1, 'Unknown': 1}}","{'Seller rating': {'4.5': 1, '5.0': 1}}","{'data3': {'Chronograph, Date': 2}}","{'data1': {'Screw-Down Crown, Luminous indices..."
Longines/ conquest/l33763877/default,l33763877,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.376.3.87.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Gold/Steel': 2, 'Grey': 1...","{'Clasp': {'Double-fold clasp': 1, 'Fold clasp...","{'Clasp material': {'Steel': 2, 'Unknown': 5}}",{'Bracelet length': {'Unknown': 7}},"{'Lug width': {'15 mm Size guide': 1, 'Unknown...",{'Buckle width': {'Unknown': 7}},{'Bracelet thickness': {'Unknown': 7}},"{'Seller rating': {'4.3': 2, '4.5': 2, '4.6': ...","{'data3': {'Date': 2, 'Unknown': 5}}","{'data1': {'Gemstones and/or diamonds': 1, 'Ge..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/Unknown/8888/default,8888,https://www.chrono24.com/longines/longines-rar...,Longines,Unknown,8888,Manual winding,Steel,Leather,1966,Good (Light signs of wear or scratches),...,"{'Bracelet color': {'Black': 1, 'Brown': 1}}",{'Clasp': {'Buckle': 2}},{'Clasp material': {'Steel': 2}},{'Bracelet length': {'190 mm (115 mm / 75 mm)'...,"{'Lug width': {'18 mm Size guide': 1, 'Unknown...","{'Buckle width': {'18 mm': 1, 'Unknown': 1}}","{'Bracelet thickness': {'2 mm': 1, 'Un

### Feature engineering

In [ ]:
ome_catalog.refnum.nunique()

1404

In [ ]:
idxcat=list(ome_catalog.index)
for yy in idxcat:
    if (ome_catalog.loc[yy,'Model'])=='Unknown':
          ome_catalog.drop([yy],axis=0,inplace=True)

In [ ]:
ome_catalog.shape

(1337, 71)

In [ ]:
ome_catalog.refnum.nunique()

1337

In [ ]:
ome_catalog3=ome_catalog.reset_index().set_index('refnum')
ome_catalog3=ome_catalog3[ome_catalog3.reset_index().groupby('refnum')['Model'].count()==1]
ome_catalog3=ome_catalog3.reset_index()
ome_catalog3=ome_catalog3.set_index('Hierarchy_code')

/tmp/ipykernel_43/3848428542.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ome_catalog3=ome_catalog3[ome_catalog3.reset_index().groupby('refnum')['Model'].count()==1]


In [ ]:
ome_catalog=ome_catalog3.copy()

In [ ]:
ome_catalog.refnum.nunique()

1337

In [ ]:
idxcat=list(ome_catalog.index)
for yy in idxcat:
    if (ome_catalog.loc[yy,'data2'])=='Unknown':
        ome_catalog.loc[yy,'data2']='No information in description'

In [ ]:
ome_catalog

,refnum,url,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color0,Clasp0,Clasp material0,Bracelet length0,Lug width0,Buckle width0,Bracelet thickness0,Seller rating0,data30,data10
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,1200,https://www.chrono24.com/longines/admiral--id2...,Longines,admiral,1200,Automatic,Yellow gold,Leather,1967,Good (Light signs of wear or scratches),...,"{'Bracelet color': {'Black': 1, 'Unknown': 2}}","{'Clasp': {'Buckle': 1, 'Unknown': 2}}","{'Clasp material': {'Gold/Steel': 1, 'Unknown'...",{'Bracelet length': {'Unknown': 3}},{'Lug width': {'Unknown': 3}},{'Buckle width': {'Unknown': 3}},{'Bracelet thickness': {'Unknown': 3}},"{'Seller rating': {' ': 2, '4.9': 1}}",{'data3': {'Unknown': 3}},{'data1': {'Unknown': 3}}
Longines/ conquest automatic/l34304076/default,l34304076,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.430.4.07.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Steel': 10, 'Unknown': 1}}","{'Clasp': {'Fold clasp': 9, 'Unknown': 2}}","{'Clasp material': {'Steel': 8, 'Unknown': 3}}",{'Bracelet length': {'Unknown': 11}},{'Lug width': {'Unknown': 11}},{'Buckle width': {'Unknown': 11}},{'Bracelet thickness': {'Unknown': 11}},"{'Seller rating': {'4.3': 1, '4.4': 1, '4.5': ...","{'data3': {'Date': 2, 'Unknown': 9}}","{'data1': {'Display back, Central seconds, Lum..."
Longines/ conquest コンクエスト/l33794/default,l33794,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.379.4,Quartz,Steel,Steel,2018,Fair (Obvious signs of wear or scratches),...,"{'Bracelet color': {'Steel': 1, 'Unknown': 1}}","{'Clasp': {'Fold clasp, hidden': 2}}",{'Clasp material': {'Steel': 2}},{'Bracelet length': {'136 mm (68 mm / 68 mm)':...,"{'Lug width': {'17 mm Size guide': 1, 'Unknown...","{'Buckle width': {'17 mm': 1, 'Unknown': 1}}","{'Bracelet thickness': {'4 mm': 1, 'Unknown': 1}}","{'Seller rating': {'4.5': 1, '5.0': 1}}","{'data3': {'Chronograph, Date': 2}}","{'data1': {'Screw-Down Crown, Luminous indices..."
Longines/ conquest/l33763877/default,l33763877,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.376.3.87.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Gold/Steel': 2, 'Grey': 1...","{'Clasp': {'Double-fold clasp': 1, 'Fold clasp...","{'Clasp material': {'Steel': 2, 'Unknown': 5}}",{'Bracelet length': {'Unknown': 7}},"{'Lug width': {'15 mm Size guide': 1, 'Unknown...",{'Buckle width': {'Unknown': 7}},{'Bracelet thickness': {'Unknown': 7}},"{'Seller rating': {'4.3': 2, '4.5': 2, '4.6': ...","{'data3': {'Date': 2, 'Unknown': 5}}","{'data1': {'Gemstones and/or diamonds': 1, 'Ge..."
Longines/ conquest/l33763887/default,l33763887,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.376.3.88.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Grey': 1, 'Steel': 1, 'Un...",{'Clasp': {'Unknown': 6}},{'Clasp material': {'Unknown': 6}},{'Bracelet length': {'Unknown': 6}},{'Lug width': {'Unknown': 6}},{'Buckle width': {'Unknown': 6}},{'Bracelet thickness': {'Unknown': 6}},"{'Seller rating': {'4.3': 2, '4.5': 2, '4.6': ...","{'data3': {'Date': 2, 'Unknown': 4}}",{'data1': {'Unknown': 6}}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/zulu time automatic 39 mm spirit/l38024532/default,l38024532,https://www.chrono24.com/longines/longines-spi...,Longines,spirit,L3.802.4.53.2,Automatic,Steel,Leather,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Brown': 7, 'Unknown': 3}}","{'Clasp': {'Buckle': 1, 'Fold clasp': 3, 'Unkn...","{'Clasp material': {'Steel': 3, 'Unknown': 7}}",{'Bracelet length': {'Unknown': 10}},{'Lug width': {'Unknown': 10}},{'Buckle width': {'Unknown': 10}},{'Bracelet thickness': {'Unknown': 10}},"{'Seller r

In [ ]:
ome_catalog[ome_catalog['data2']=='Unknown']

,refnum,url,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color0,Clasp0,Clasp material0,Bracelet length0,Lug width0,Buckle width0,Bracelet thickness0,Seller rating0,data30,data10
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,


In [ ]:
ome_catalog['data2']

Hierarchy_code
Longines/ admiral automatic/1200/default                         Small.mark.on glass 5 o'clock Perfectly working
Longines/ conquest automatic/l34304076/default                                     No information in description
Longines/ conquest コンクエスト/l33794/default                       This is a pre-owned Longines Conquest L3.379.4...
Longines/ conquest/l33763877/default                                               No information in description
Longines/ conquest/l33763887/default                                               No information in description
                                                                                     ...                        
Longines/zulu time automatic 39 mm spirit/l38024532/default                        No information in description
Longines/アドミラル 5スター デイデイト 自動巻き/l76342/default                  Brand : Longines Model : Admiral - L76342 - Au...
Longines/conquest/9002/default                                                   

In [ ]:
list(ome_catalog['data1'])

['Small seconds',
 'Display back, Central seconds, Luminous hands, Screw-Down Crown, Luminous indices',
 'Screw-Down Crown, Luminous indices',
 'Gemstones and/or diamonds',
 'Small seconds',
 'Display back, Central seconds, Luminous hands, Screw-Down Crown, Luminous indices',
 'Small seconds',
 'Small seconds',
 'Gemstones and/or diamonds',
 'Small seconds',
 'Small seconds',
 'Small seconds',
 'Small seconds',
 'Display back, Chronometer',
 'Small seconds',
 'Only Original Parts',
 'Small seconds',
 'Small seconds, Screw-Down Crown',
 'Luminous hands, Luminous indices',
 'Small seconds',
 'Small seconds',
 'Central seconds',
 'Small seconds',
 'Small seconds',
 'Small seconds',
 'Display back, Central seconds, Luminous hands, Screw-Down Crown, Luminous indices',
 'Display back, Central seconds, Screw-Down Crown',
 'Small seconds',
 'Small seconds, Luminous hands, Screw-Down Crown, Luminous indices',
 'Central seconds, Small seconds, Luminous hands, Screw-Down Crown, Luminous indices',

In [ ]:
ome_catalog.drop(['NormalizedPrice','Dissimilarity'],axis=1,inplace=True)

In [ ]:
ome_catalog.rename(columns={'data2':'Description'},inplace=True)

In [ ]:
ome_catalog['data1']=[x.split(', ')  for x in list(ome_catalog['data1'])]

In [ ]:
ome_catalog.loc[:,'data1']=[set(x)  for x in list(ome_catalog['data1'])]

In [ ]:
setunion=set([])
for rr in list(ome_catalog['data1']):
    setunion=setunion.union(rr)

In [ ]:
maxlen=0
maxset=set([])
for rr in list(ome_catalog['data1']):
    if len(rr)>maxlen:
        maxlen=len(rr)
        maxset=rr
print(maxlen)
print(maxset)

10
{'Luminous indices', 'Screw-Down Crown', 'Only Original Parts', 'Chronometer', 'Small seconds', 'Master Chronometer', 'Luminous hands', 'Rotating Bezel', 'Central seconds', 'Gemstones and/or diamonds'}


In [ ]:
len(setunion)

20

In [ ]:
ome_catalog[['data1']]

,data1
Hierarchy_code,
Longines/ admiral automatic/1200/default,{Small seconds}
Longines/ conquest automatic/l34304076/default,"{Luminous indices, Screw-Down Crown, Luminous ..."
Longines/ conquest コンクエスト/l33794/default,"{Luminous indices, Screw-Down Crown}"
Longines/ conquest/l33763877/default,{Gemstones and/or diamonds}
Longines/ conquest/l33763887/default,{Small seconds}
...,...
Longines/zulu time automatic 39 mm spirit/l38024532/default,"{Chronometer, Luminous indices, Screw-Down Cro..."
Longines/アドミラル 5スター デイデイト 自動巻き/l76342/default,{Small seconds}
Longines/conquest/9002/default,{Small seconds}


In [ ]:
mlb = MultiLabelBinarizer()
data1_ext=mlb.fit_transform(list(ome_catalog['data1']))
data1_ext_=pd.DataFrame(data1_ext,columns=mlb.classes_,index=ome_catalog.index)
data1_ext_.apply('sum',axis=1)[0:50]

Hierarchy_code
Longines/ admiral automatic/1200/default                                         1
Longines/ conquest automatic/l34304076/default                                   5
Longines/ conquest コンクエスト/l33794/default                                         2
Longines/ conquest/l33763877/default                                             1
Longines/ conquest/l33763887/default                                             1
Longines/ conquest/l34304726/default                                             5
Longines/ presence/l43214112/default                                             1
Longines/ presence/l43221117/default                                             1
Longines/ プリマルナ pgコンビ d ssxpg クォーツ/l81095796/default                             1
Longines/ メンズ /l56774/default                                                    1
Longines/1 100th st moritz conquest/l37004786/default                            1
Longines/20 8mm x 32mm ladies watch dolce vita longines/l5255471b/defaul

In [ ]:
data1_ext_.replace({1:'Yes',0:'No'},inplace=True)

In [ ]:
data1_ext_

,,Central seconds,Chronometer,Display back,Gemstones and/or diamonds,Genevian Seal,Guilloché dial,Guilloché dial (handwork),Luminous hands,Luminous indices,Luminous numerals,Master Chronometer,Only Original Parts,PVD/DLC coating,Power Reserve Display,Quick Set,Rotating Bezel,Screw-Down Crown,Small seconds,Tempered blue hands
Hierarchy_code,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,No
Longines/ conquest automatic/l34304076/default,No,Yes,No,Yes,No,No,No,No,Yes,Yes,No,No,No,No,No,No,No,Yes,No,No
Longines/ conquest コンクエスト/l33794/default,No,No,No,No,No,No,No,No,No,Yes,No,No,No,No,No,No,No,Yes,No,No
Longines/ conquest/l33763877/default,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
Longines/ conquest/l33763887/default,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/zulu time automatic 39 mm spirit/l38024532/default,No,No,Yes,No,No,No,No,No,Yes,Yes,No,No,No,No,No,No,No,Yes,No,No
Longines/アドミラル 5スター デイデイト 自動巻き/l76342/default,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,No
Longines/conquest/9002/default,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,No


In [ ]:
list(ome_catalog.columns)

['refnum',
 'url',
 'Brand',
 'Model',
 'Reference number',
 'Movement',
 'Case material',
 'Bracelet material',
 'Year of production',
 'Condition',
 'Scope of delivery',
 'Gender',
 'Location',
 'Availability',
 'Caliber/movement',
 'Power reserve',
 'Number of jewels',
 'Frequency',
 'Base caliber',
 'Case diameter',
 'Water resistance',
 'Crystal',
 'Dial',
 'Thickness',
 'Bezel material',
 'Dial numerals',
 'Bracelet color',
 'Clasp',
 'Clasp material',
 'Bracelet length',
 'Lug width',
 'Buckle width',
 'Bracelet thickness',
 'Description',
 'Seller rating',
 'data3',
 'data1',
 'PriceUSDthou',
 'Movement0',
 'Case material0',
 'Bracelet material0',
 'Year of production0',
 'Condition0',
 'Scope of delivery0',
 'Gender0',
 'Location0',
 'Availability0',
 'Caliber/movement0',
 'Power reserve0',
 'Number of jewels0',
 'Frequency0',
 'Base caliber0',
 'Case diameter0',
 'Water resistance0',
 'Crystal0',
 'Dial0',
 'Thickness0',
 'Bezel material0',
 'Dial numerals0',
 'Bracelet color

In [ ]:
list(ome_catalog.columns)[:list(ome_catalog.columns).index('Movement0')]

['refnum',
 'url',
 'Brand',
 'Model',
 'Reference number',
 'Movement',
 'Case material',
 'Bracelet material',
 'Year of production',
 'Condition',
 'Scope of delivery',
 'Gender',
 'Location',
 'Availability',
 'Caliber/movement',
 'Power reserve',
 'Number of jewels',
 'Frequency',
 'Base caliber',
 'Case diameter',
 'Water resistance',
 'Crystal',
 'Dial',
 'Thickness',
 'Bezel material',
 'Dial numerals',
 'Bracelet color',
 'Clasp',
 'Clasp material',
 'Bracelet length',
 'Lug width',
 'Buckle width',
 'Bracelet thickness',
 'Description',
 'Seller rating',
 'data3',
 'data1',
 'PriceUSDthou']

In [ ]:
ome_catalog_left=ome_catalog.drop(list(ome_catalog.columns)[list(ome_catalog.columns).index(firstcolnameendwith0(ome_catalog)):],axis=1)

In [ ]:
ome_catalog_right=ome_catalog.drop(list(ome_catalog.columns)[:list(ome_catalog.columns).index(firstcolnameendwith0(ome_catalog))],axis=1)

In [ ]:
ome_catalog=pd.concat([ome_catalog_left,data1_ext_,ome_catalog_right],axis=1)

In [ ]:
ome_catalog.drop(['data1'],axis=1,inplace=True)

In [ ]:
ome_catalog.drop([''],axis=1,errors='ignore',inplace=True)

In [ ]:
ome_catalog.rename(columns={'data3':'Complications'},inplace=True)

In [ ]:
#refnum_list_calibdrop=[]
#for tt in list(ome_catalog.index):
#    v=ome_catalog.loc[tt,'Base caliber0']
#    vlst=[]
#    for key,value in v.items():
#        vlst.append(key)
#    if 'Unknown' in vlst:
#      if len(vlst)==1:
#         w=ome_catalog.loc[tt,'Caliber/movement0']
#         wlst=[]
#         for key,value in w.items():
#            wlst.append(key)
#         if 'Unknown' in wlst:
#            if len(wlst)==1:
#               refnum_list_calibdrop.append(ome_catalog.loc[tt,'refnum'])
#               print(ome_catalog.loc[tt,'refnum'])
#pd.DataFrame(refnum_list_calibdrop).to_csv(f"refnum_list_calibdrop{BRND.lower()}v1.csv")


### Catalog saving

In [ ]:
ome_catalog

,refnum,url,Brand,Model,Reference number,Movement,Case material,Bracelet material,Year of production,Condition,...,Bracelet color0,Clasp0,Clasp material0,Bracelet length0,Lug width0,Buckle width0,Bracelet thickness0,Seller rating0,data30,data10
Hierarchy_code,,,,,,,,,,,,,,,,,,,,,
Longines/ admiral automatic/1200/default,1200,https://www.chrono24.com/longines/admiral--id2...,Longines,admiral,1200,Automatic,Yellow gold,Leather,1967,Good (Light signs of wear or scratches),...,"{'Bracelet color': {'Black': 1, 'Unknown': 2}}","{'Clasp': {'Buckle': 1, 'Unknown': 2}}","{'Clasp material': {'Gold/Steel': 1, 'Unknown'...",{'Bracelet length': {'Unknown': 3}},{'Lug width': {'Unknown': 3}},{'Buckle width': {'Unknown': 3}},{'Bracelet thickness': {'Unknown': 3}},"{'Seller rating': {' ': 2, '4.9': 1}}",{'data3': {'Unknown': 3}},{'data1': {'Unknown': 3}}
Longines/ conquest automatic/l34304076/default,l34304076,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.430.4.07.6,Automatic,Steel,Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Steel': 10, 'Unknown': 1}}","{'Clasp': {'Fold clasp': 9, 'Unknown': 2}}","{'Clasp material': {'Steel': 8, 'Unknown': 3}}",{'Bracelet length': {'Unknown': 11}},{'Lug width': {'Unknown': 11}},{'Buckle width': {'Unknown': 11}},{'Bracelet thickness': {'Unknown': 11}},"{'Seller rating': {'4.3': 1, '4.4': 1, '4.5': ...","{'data3': {'Date': 2, 'Unknown': 9}}","{'data1': {'Display back, Central seconds, Lum..."
Longines/ conquest コンクエスト/l33794/default,l33794,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.379.4,Quartz,Steel,Steel,2018,Fair (Obvious signs of wear or scratches),...,"{'Bracelet color': {'Steel': 1, 'Unknown': 1}}","{'Clasp': {'Fold clasp, hidden': 2}}",{'Clasp material': {'Steel': 2}},{'Bracelet length': {'136 mm (68 mm / 68 mm)':...,"{'Lug width': {'17 mm Size guide': 1, 'Unknown...","{'Buckle width': {'17 mm': 1, 'Unknown': 1}}","{'Bracelet thickness': {'4 mm': 1, 'Unknown': 1}}","{'Seller rating': {'4.5': 1, '5.0': 1}}","{'data3': {'Chronograph, Date': 2}}","{'data1': {'Screw-Down Crown, Luminous indices..."
Longines/ conquest/l33763877/default,l33763877,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.376.3.87.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Gold/Steel': 2, 'Grey': 1...","{'Clasp': {'Double-fold clasp': 1, 'Fold clasp...","{'Clasp material': {'Steel': 2, 'Unknown': 5}}",{'Bracelet length': {'Unknown': 7}},"{'Lug width': {'15 mm Size guide': 1, 'Unknown...",{'Buckle width': {'Unknown': 7}},{'Bracelet thickness': {'Unknown': 7}},"{'Seller rating': {'4.3': 2, '4.5': 2, '4.6': ...","{'data3': {'Date': 2, 'Unknown': 5}}","{'data1': {'Gemstones and/or diamonds': 1, 'Ge..."
Longines/ conquest/l33763887/default,l33763887,https://www.chrono24.com/longines/longines-con...,Longines,conquest,L3.376.3.88.7,Quartz,Steel,Gold/Steel,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Grey': 1, 'Steel': 1, 'Un...",{'Clasp': {'Unknown': 6}},{'Clasp material': {'Unknown': 6}},{'Bracelet length': {'Unknown': 6}},{'Lug width': {'Unknown': 6}},{'Buckle width': {'Unknown': 6}},{'Bracelet thickness': {'Unknown': 6}},"{'Seller rating': {'4.3': 2, '4.5': 2, '4.6': ...","{'data3': {'Date': 2, 'Unknown': 4}}",{'data1': {'Unknown': 6}}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Longines/zulu time automatic 39 mm spirit/l38024532/default,l38024532,https://www.chrono24.com/longines/longines-spi...,Longines,spirit,L3.802.4.53.2,Automatic,Steel,Leather,2023,"New (Brand new, without any signs of wear)",...,"{'Bracelet color': {'Brown': 7, 'Unknown': 3}}","{'Clasp': {'Buckle': 1, 'Fold clasp': 3, 'Unkn...","{'Clasp material': {'Steel': 3, 'Unknown': 7}}",{'Bracelet length': {'Unknown': 10}},{'Lug width': {'Unknown': 10}},{'Buckle width': {'Unknown': 10}},{'Bracelet thickness': {'Unknown': 10}},"{'Seller r

In [ ]:
ome_catalog.to_excel(f"{BRND.lower()}_cataloguev1.xlsx")